# CS781 Capstone — Phase 6: Abstractive Legal Summarization on 2×T4

**Kaggle settings:** GPU T4 ×2, Internet **on**, secret **`HF`** or **`HF_TOKEN`** (ticked). **Add Input:** your private
Phase-1 model datasets (A, B2, C+D, optionally B3 = `ctxext/best_newctx.pt`) and, from session 2 on, earlier outputs of
this notebook (either account) so finished runs (teachers, stage-1 files) are reused.

**Plumbing (declare it):** head+tail source selection (`src_len` tokens) → target = reference truncated to `tgt_len`
(+ `</s>`) → Phase-1 LM with its tied head + your technique → token cross-entropy on the summary. A/B/D read
`[source][sep][summary]` (D bidirectional over source + sep), C encodes the source and decodes the summary. Hierarchical:
a finished run writes stage-1 chunk summaries (`after.stage1`), a stage-2 run (`source.type = "stage1"`) learns
chunk summaries → reference. Checkpoint = lowest validation loss on a seeded hold-out of train.json; summaries for
validation (ROUGE-1/L + BERTScore roberta-large, local proxy) and test → `prediction.json` (list format) + zip.

**Your parts (course rule 9):** `techniques/*.py`, `kd.py`, and every value in `experiments.py` (and the decoding lists in
Stage 7). Outputs: `/kaggle/working/phase6/runs/<name>/`, zips in `/kaggle/working/submissions/`.

## Stage 0 — setup

In [ ]:
import os, sys, json, glob, time, hashlib, subprocess, shutil, inspect
SESSION_START = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score", "bert-score"], check=True)
WORK = "/kaggle/working/phase6"
for d in ("phase1_models", "techniques", "configs", "runs", "data", "cache", "sweeps"):
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
import torch
N_GPUS = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(N_GPUS)])

## Stage 1 — plumbing files (declare)

In [ ]:
%%writefile /kaggle/working/phase6/a1_tokenizer.py
# Legal byte-level BPE (your A1 tokenizer, optimised)
# tokenizer.py
#
# Byte-level BPE with an end-of-word marker (A1 design), adapted for the CS781 legal corpus.
#
# Changes vs. the A1 version (original kept in Assignment3/a1_tokenizer.py):
#   * Tokens are integer ids backed by raw bytes, not strings. Previously the byte "<", "/",
#     "w", ">" could merge into a token identical to the "</w>" marker (or to "<pad>"), and
#     decode() would then insert spurious spaces. Ids make that collision impossible.
#   * Legal-text pre-tokenisation inside each whitespace word: letter runs, digit groups of
#     at most 3, and punctuation runs are merged separately ("s.302(1)," -> s . 302 ( 1 ) ,),
#     so "court", "court," and "court." share one vocabulary entry instead of three, and
#     years / section numbers / citations don't flood the vocabulary. Only the last piece of
#     each whitespace word carries </w>, so decode(encode(x)) == " ".join(x.split()).
#   * Incremental BPE training (pair counts + inverted index + lazy max-heap): each merge
#     only touches the words that contain the merged pair, instead of recounting every pair
#     in the corpus for every merge. 32k merges on a legal sample take minutes, not days.
#   * Encoding uses merge ranks (GPT-2 style, same result as applying merges in order) and a
#     bounded per-word cache (the old lru_cache on a method also cached `self` and never freed).
#   * Extra reserved special tokens for later phases (cls/sep/mask, span-corruption sentinels).
#   * encode_many() encodes many documents in parallel worker processes.
#
# Assignment 3 contract is unchanged: train_tokenizer(train_text, vocab_size) returns an object
# with encode(text) -> list[int], decode(ids) -> str and a vocab_size attribute.

import re
import os
import heapq
import string
import json
from collections import Counter, defaultdict


# ============================================================
# SPECIAL TOKENS
# ============================================================

N_SENTINELS = 25

SPECIALS = (
    ["<pad>", "<unk>", "<s>", "</s>", "<cls>", "<sep>", "<mask>"]
    + [f"<extra_id_{i}>" for i in range(N_SENTINELS)]
)   # 32 specials: ids 0..31

MIN_COUNT = 2

# Pieces inside one whitespace word: letter runs | digit groups (<= 3) | punctuation runs | underscores.
# The alternatives cover every non-space character, so "".join(pieces) == word always holds.
PRETOKENIZE = re.compile(r"[^\W\d_]+|\d{1,3}|[^\w\s]+|_+")

MAX_CACHE_WORDS = 2_000_000


# ============================================================
# TEXT CLEANING (A1 helpers, unchanged; not used by the BPE)
# ============================================================

def check_valid_en(word):
    is_en = False
    is_hi = False
    has_digit = False

    for ch in word:
        code = ord(ch)

        # English alphabet
        if 0x0041 <= code <= 0x005A or 0x0061 <= code <= 0x007A:
            is_en = True

        # Arabic numerals
        elif 0x0030 <= code <= 0x0039:
            has_digit = True

        # Devanagari numerals
        elif 0x0966 <= code <= 0x096F:
            has_digit = True

        # Devanagari
        elif 0x0900 <= code <= 0x097F:
            is_hi = True

    if has_digit:
        return False

    return is_en and not is_hi


def clean_text_and_tokenize_en(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    return [word for word in tokens if check_valid_en(word)]


# ============================================================
# BYTE-LEVEL BPE
# ============================================================

def split_word(word):
    """Whitespace word -> list of (piece, is_last_piece)."""
    pieces = PRETOKENIZE.findall(word) or [word]
    last = len(pieces) - 1
    return [(p, i == last) for i, p in enumerate(pieces)]


class ByteBPE:

    def __init__(self, vocab_size=16000):

        self.vocab_size = vocab_size

        # Id layout: specials | 256 bytes | </w> | one id per learned merge
        self.n_special = len(SPECIALS)
        self.byte_offset = self.n_special
        self.eow_id = self.byte_offset + 256

        self.merges = []            # list of (left_id, right_id); merge k creates id eow_id + 1 + k
        self._reset_tables()

    # --------------------------------------------------------
    # VOCAB TABLES
    # --------------------------------------------------------

    def _reset_tables(self):
        self.token_bytes = [b""] * self.n_special + [bytes([b]) for b in range(256)] + [b""]
        self.token_eow = [False] * (self.n_special + 256) + [True]
        for left, right in self.merges:
            self.token_bytes.append(self.token_bytes[left] + self.token_bytes[right])
            self.token_eow.append(self.token_eow[right])
        self.ranks = {pair: rank for rank, pair in enumerate(self.merges)}
        self.merge_to_id = {pair: self.eow_id + 1 + rank for rank, pair in enumerate(self.merges)}
        # decode table: bytes of each token, plus a space if it ends a word; specials decode to nothing
        self._decode_table = [
            b"" if i < self.n_special else tb + (b" " if eow else b"")
            for i, (tb, eow) in enumerate(zip(self.token_bytes, self.token_eow))
        ]
        self.itos = list(SPECIALS) + [self._display(i) for i in range(self.n_special, len(self.token_bytes))]
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.vocab = self.itos
        self._cache = {}

    def _display(self, idx):
        """Human-readable token string (for inspection only; never used for encoding)."""
        text = self.token_bytes[idx].decode("utf-8", errors="backslashreplace")
        return text + ("</w>" if self.token_eow[idx] else "")

    @property
    def special_ids(self):
        return {tok: i for i, tok in enumerate(SPECIALS)}

    @property
    def eot_id(self):
        """Document separator used in the pretraining stream."""
        return SPECIALS.index("</s>")

    def get_vocab_size(self):
        return len(self.token_bytes)

    # --------------------------------------------------------
    # PIECE -> INITIAL SYMBOLS
    # --------------------------------------------------------

    def _piece_symbols(self, piece, is_last):
        symbols = [self.byte_offset + b for b in piece.encode("utf-8")]
        if is_last:
            symbols.append(self.eow_id)
        return symbols

    # --------------------------------------------------------
    # TRAIN BPE
    # --------------------------------------------------------

    @staticmethod
    def count_words(texts):
        """Whitespace-word frequencies of an iterable of texts."""
        counts = Counter()
        for text in texts:
            counts.update(str(text).split())
        return counts

    def fit(self, tokenized_corpus=None, target_vocab_size=None, min_count=MIN_COUNT,
            word_counts=None, min_word_freq=1, verbose=True):
        """Learn merges.

        tokenized_corpus: iterable of lists of whitespace words (A1/A3 interface), or
        word_counts:      a precomputed {whitespace word: frequency} mapping (faster for big corpora).
        min_word_freq:    ignore whitespace words rarer than this while learning merges (speed knob).
        """
        target = self.vocab_size if target_vocab_size is None else target_vocab_size
        self.vocab_size = target

        if word_counts is None:
            word_counts = Counter()
            for sentence_tokens in tokenized_corpus:
                word_counts.update(sentence_tokens)

        # whitespace words -> (piece, is_last) frequencies
        piece_freq = Counter()
        for word, count in word_counts.items():
            if count < min_word_freq:
                continue
            for piece in split_word(word):
                piece_freq[piece] += count

        words = [self._piece_symbols(p, last) for p, last in piece_freq]
        freqs = list(piece_freq.values())
        del piece_freq

        if verbose:
            print("=" * 60)
            print("BUILDING BYTE-LEVEL BPE")
            print("=" * 60)
            print(f"Unique pre-token pieces: {len(words):,}")
            print(f"Initial vocabulary size: {self.eow_id + 1:,}")
            print(f"Target vocabulary size: {target:,}")

        # pair -> total count, pair -> set of word indices containing it
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for wi, (symbols, f) in enumerate(zip(words, freqs)):
            for pair in zip(symbols, symbols[1:]):
                pair_counts[pair] += f
                where[pair].add(wi)

        # max-heap on count with deterministic tie-break on ids; stale entries are skipped lazily
        heap = [(-c, pair) for pair, c in pair_counts.items()]
        heapq.heapify(heap)

        self.merges = []
        next_id = self.eow_id + 1

        while next_id < target and heap:
            neg, pair = heapq.heappop(heap)
            current = pair_counts.get(pair, 0)
            if current != -neg:
                if current > 0:
                    heapq.heappush(heap, (-current, pair))
                continue
            if current < min_count:
                if verbose:
                    print(f"\nStopping: best pair frequency = {current}")
                break

            left, right = pair
            new_id = next_id
            next_id += 1
            self.merges.append(pair)
            touched = Counter()

            for wi in list(where.pop(pair, ())):
                symbols, f = words[wi], freqs[wi]
                # remove this word's old pairs
                for p in zip(symbols, symbols[1:]):
                    pair_counts[p] -= f
                    touched[p] += 0
                    s = where.get(p)
                    if s is not None:
                        s.discard(wi)
                # merge left-to-right
                merged, i, n = [], 0, len(symbols)
                while i < n:
                    if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                        merged.append(new_id)
                        i += 2
                    else:
                        merged.append(symbols[i])
                        i += 1
                words[wi] = merged
                # add the new pairs
                for p in zip(merged, merged[1:]):
                    pair_counts[p] += f
                    touched[p] += 0
                    where[p].add(wi)

            pair_counts.pop(pair, None)
            for p in touched:
                c = pair_counts.get(p, 0)
                if c <= 0:
                    pair_counts.pop(p, None)
                    where.pop(p, None)
                elif new_id in p:
                    heapq.heappush(heap, (-c, p))   # only pairs with the new symbol can grow

            if verbose and len(self.merges) % 1000 == 0:
                print(f"Merges: {len(self.merges):,} | Vocab: {next_id:,} | Latest pair count: {current:,}")

        self._reset_tables()

        if verbose:
            print("\n" + "=" * 60)
            print("BPE TRAINING COMPLETE")
            print("=" * 60)
            print(f"Final vocabulary size: {self.get_vocab_size():,}")
            print(f"Total merges learned: {len(self.merges):,}")

        return self

    # --------------------------------------------------------
    # ENCODE
    # --------------------------------------------------------

    def _bpe(self, symbols):
        """Repeatedly merge the lowest-rank adjacent pair (== applying merges in learned order)."""
        ranks = self.ranks
        while len(symbols) > 1:
            best, best_rank = None, None
            for pair in zip(symbols, symbols[1:]):
                r = ranks.get(pair)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = pair, r
            if best is None:
                break
            new_id = self.merge_to_id[best]
            left, right = best
            merged, i, n = [], 0, len(symbols)
            while i < n:
                if i < n - 1 and symbols[i] == left and symbols[i + 1] == right:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(symbols[i])
                    i += 1
            symbols = merged
        return symbols

    def encode_word(self, word):
        ids = self._cache.get(word)
        if ids is None:
            ids = []
            for piece, is_last in split_word(word):
                ids.extend(self._bpe(self._piece_symbols(piece, is_last)))
            ids = tuple(ids)
            if len(self._cache) >= MAX_CACHE_WORDS:
                self._cache.clear()
            self._cache[word] = ids
        return ids

    def encode_tokens(self, sentence_tokens):
        ids = []
        for word in sentence_tokens:
            ids.extend(self.encode_word(word))
        return ids

    def encode(self, text):
        """text -> list of integer ids (whitespace is normalised to single spaces)."""
        cache = self._cache
        ids = []
        for word in str(text).split():
            w = cache.get(word)
            if w is None:
                w = self.encode_word(word)
            ids.extend(w)
        return ids

    def pool(self, workers=None):
        """Process pool whose workers each hold a copy of this tokenizer (and keep their word cache).
        Use with the module-level worker functions, e.g. pool.imap(encode_array_worker, texts)."""
        import multiprocessing as mp
        workers = workers or os.cpu_count() or 1
        return mp.get_context("fork").Pool(workers, initializer=_worker_init, initargs=(self,))

    def encode_many(self, texts, workers=None, chunksize=64):
        """Encode many documents in parallel processes; yields lists of ids in input order."""
        with self.pool(workers) as pool:
            yield from pool.imap(_worker_encode, texts, chunksize=chunksize)

    # --------------------------------------------------------
    # DECODE
    # --------------------------------------------------------

    def decode(self, ids):
        table = self._decode_table
        n = len(table)
        out = b"".join(table[int(i)] if 0 <= int(i) < n else b"" for i in ids)
        return out.decode("utf-8", errors="replace").strip()

    def ends_word(self, idx):
        """True if token idx closes a whitespace word (used to count generated words)."""
        return self.token_eow[idx]

    # --------------------------------------------------------
    # SAVE / LOAD
    # --------------------------------------------------------

    def to_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "format": "legal-bytebpe-v2",
                    "specials": SPECIALS,
                    "merges": self.merges,
                    "vocab_size": self.get_vocab_size(),
                    "itos": self.itos,          # readable, informational only
                },
                f,
                ensure_ascii=False,
            )
        print(f"Saved tokenizer to: {path}")

    @classmethod
    def from_json(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        if data.get("format") != "legal-bytebpe-v2":
            raise ValueError(f"{path} was not written by this tokenizer version")
        if data["specials"] != SPECIALS:
            raise ValueError("special tokens differ from this tokenizer version")
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = [tuple(pair) for pair in data["merges"]]
        tokenizer._reset_tables()
        return tokenizer


# worker-process helpers for encode_many (module level so they pickle)
_WORKER_TOKENIZER = None


def _worker_init(tokenizer):
    global _WORKER_TOKENIZER
    _WORKER_TOKENIZER = tokenizer


def _worker_encode(text):
    return _WORKER_TOKENIZER.encode(text)


def encode_array_worker(text):
    """(uint16 id array, whitespace word count) for one document; for tokenizer.pool().imap."""
    import numpy as np
    return np.asarray(_WORKER_TOKENIZER.encode(text), dtype=np.uint16), text.count(" ") + 1


# ============================================================
# ASSIGNMENT 3 TRAINING FUNCTION
# ============================================================

def train_tokenizer(train_text: str, vocab_size: int):
    text = str(train_text)
    print(f"Training tokenizer on {len(text):,} characters")
    tokenizer = ByteBPE(vocab_size=vocab_size)
    tokenizer.fit(
        word_counts=ByteBPE.count_words(text.splitlines()),
        target_vocab_size=vocab_size,
        min_count=MIN_COUNT,
    )
    return tokenizer


# ============================================================
# SIMPLE SMOKE TEST
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("TOKENIZER SMOKE TEST")
    print("=" * 60)

    train_text = """
    the court has jurisdiction over this matter under Section 302(1) of the Code.
    the court shall consider the evidence, vide AIR 1966 SC 1882, dated 18.12.1961.
    the evidence presented before the court </w> <pad> was 12,345 pages.
    """

    tokenizer = train_tokenizer(train_text, vocab_size=400)

    for test_text in [
        "the court has jurisdiction",
        "Section 302(1), AIR 1966 SC 1882 </w><pad> naïve — ₹5,00,000/- (u/s 138)",
    ]:
        ids = tokenizer.encode(test_text)
        decoded = tokenizer.decode(ids)
        print("\nInput:  ", test_text)
        print("Ids:    ", ids)
        print("Pieces: ", [tokenizer.itos[i] for i in ids])
        print("Decoded:", decoded)
        assert decoded == " ".join(test_text.split()), "round trip failed"

    print("\nVocabulary size:", tokenizer.get_vocab_size())

    tokenizer.to_json("bpe_smoke_test.json")
    loaded = ByteBPE.from_json("bpe_smoke_test.json")
    assert loaded.encode(test_text) == tokenizer.encode(test_text)
    os.remove("bpe_smoke_test.json")

    print("\n" + "=" * 60)
    print("SMOKE TEST PASSED")
    print("=" * 60)

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/rope.py
# your rope.py (two lines fixed, marked FIX (Claude))
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/arch1.py
# Architecture 1 — attention1.py + decoder1.py (your code)
# ======== attention1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Multi-head causal self-attention, implemented from raw tensor ops."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # Hook for Q1.4. None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        # TODO: define self.q_proj, self.k_proj, self.v_proj, self.out_proj as nn.Linear
        # (no bias). All should map d_model -> d_model.
        self.q_proj = nn.Linear(d_model,d_model,bias=False)
        self.k_proj = nn.Linear(d_model,d_model,bias=False)
        self.v_proj = nn.Linear(d_model,d_model,bias=False)
        self.out_proj = nn.Linear(d_model,d_model,bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # TODO: register a (max_seq_len, max_seq_len) lower-triangular boolean mask as a
        # buffer named `causal_mask` (True = allowed to attend, False = masked out), so
        # you don't rebuild it on every forward call.
        causal_mask = torch.tril(
            torch.ones(max_seq_len,max_seq_len,dtype=torch.bool)
        )
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        # TODO 1: project to q, k, v  (each (B, T, C))
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        # TODO 2: reshape each to (B, n_heads, T, d_head) — split C into (n_heads, d_head)
        #         and move n_heads next to the batch dim.
        q = torch.reshape(q,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        k = torch.reshape(k,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        v = torch.reshape(v,(B,T,self.n_heads,self.d_head)).transpose(1,2)
        
        # TODO 3: compute attention scores = (q @ k^T) * self.attention_scale
        #         -> (B, n_heads, T, T). Do NOT hard-code 1/sqrt(d_head): Q1.4
        #         changes this hook for the scaling ablation.
        attention_scores = (q@k.transpose(2,3))*self.attention_scale
        # TODO 4: apply the causal mask (set disallowed positions to -inf before softmax)
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
        # TODO 5: softmax over the last dim, apply self.attn_dropout
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
        # TODO 6: weighted sum with v -> (B, n_heads, T, d_head)
        weighted_sum = attention_weights@v
        # TODO 7: reshape back to (B, T, C) and apply self.out_proj + self.resid_dropout
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder1.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM, built from scratch (no nn.Transformer*)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 512
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4          # == n_heads for plain MHA; used in Question 2
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # or "rope", used in Question 3
    attention_scale: float = None      # Q1.4 hook; None -> 1/sqrt(d_head)
    norm_first: bool = False            # norm_first=True is for pre-norm else post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        # TODO: two linear layers with a GELU in between, plus dropout
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """
    Pre-norm Transformer block: x + Attn(LN(x)), then x + FFN(LN(x)).
    OR
    Post-norm Transformer block: LN(x + Attn(x)), then LN(x + FFN(x)).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        # TODO: implement pre-norm when norm_first=True,
        # cfg.norm_first = true for pre norm
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    """The model class the autograder imports directly. Do not rename it."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # (RoPE is applied inside attention instead of here — see Question 3)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/arch3.py
# Architecture 2 option — rope.py + attention3.py + decoder3.py (your code: GQA + RoPE)
# ======== rope.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    # TODO: compute inverse frequencies: 1 / base^(2i/d_head) for i in [0, d_head/2)
    inverse = [1/(base**(2*i/d_head)) for i in range(0,d_head//2)]
    inverse = torch.tensor(inverse, dtype=torch.float32)  # FIX (Claude): torch.outer needs a tensor, not a list
    # TODO: outer product of positions [0..max_seq_len) with inverse frequencies -> (T, d_head/2)
    pos = torch.arange(max_seq_len, dtype=torch.float32)  # FIX (Claude): float, to match `inverse` in torch.outer
    outer = torch.outer(pos,inverse)
    # TODO: construct angles = torch.cat([freqs, freqs], dim=-1) so the shape
    #       becomes (T, d_head) and matches the split-half rotate_half convention.
    #       Do NOT use repeat_interleave here; it produces a different channel layout.
    #       Then return angles.cos(), angles.sin().
    angles = torch.cat([outer,outer],dim=-1)
    return angles.cos() , angles.sin()
    


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    x1,x2 = torch.chunk(x,2,dim=-1)
    x2=-x2
    x = torch.cat([x2,x1],dim=-1)
    return x



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = torch.unsqueeze(cos,dim=0)
    cos = torch.unsqueeze(cos,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    sin = torch.unsqueeze(sin,dim=0)
    x_rotated = x*cos+rotate_half(x)*sin
    
    return x_rotated


# ======== attention3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output


# ======== decoder3.py (your A3 code; cross-file import removed, FIX lines marked) ========
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/attention3.py
# your attention3.py (verbatim)
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
        attention_scores = attention_scores.masked_fill(
            ~self.causal_mask[:T,:T],float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/non_causal_self_attention.py
# your non_causal_self_attention.py (verbatim)
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/cross_attention.py
# your cross_attention.py (verbatim)
"""Cross Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x , memory):
        B, T, C = x.shape
        _,S,_ = memory.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale
       
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/encoder.py
# your encoder.py (verbatim)
"""Encoder-Block of  Transformer LM -- Block now passes use_rope through to attention, and
EncoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from non_causal_self_attention import SelfAttention


@dataclass
class EncLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class EncBlock(nn.Module):
    def __init__(self, cfg: EncLMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = SelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class EncoderLM(nn.Module):
    def __init__(self, cfg: EncLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        #self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        #self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        #x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/decoder4.py
# your decoder4.py (verbatim)
"""Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2)."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention3 import CausalSelfAttention
from cross_attention import CrossAttention


@dataclass
class DecLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class DecBlock(nn.Module):
    def __init__(self, cfg: DecLMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.crattn = CrossAttention(cfg.d_model , cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,
        cfg.dropout,use_rope=False,attention_scale=cfg.attention_scale)

        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ln3 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x , memory):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1)+x
            y3 = self.ln2(y2)
            y4 = self.crattn(y3,memory)+y2
            y5 = self.ln3(y4)
            y6 = self.ffn(y5)+y4
            output=y6
        else:
            y1 = self.attn(x)
            y2 = self.ln1(y1+x)
            y3 = self.crattn(y2,memory)
            y4 = self.ln2(y3+y2)
            y5 = self.ffn(y4)
            y6 = self.ln3(y5+y4)
            output=y6
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: DecLMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([DecBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor , memory:torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x,memory)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/enc_dec.py
# your enc_dec.py (verbatim)
from dataclasses import dataclass 
import torch  
import torch.nn as nn  
from encoder import EncBlock, sinusoidal_positional_encoding  
from decoder4 import DecBlock

@dataclass
class EncLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


@dataclass
class DecLMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False


class EncDecLM(nn.Module):
    def __init__(self,cfg1:EncLMConfig,cfg2:DecLMConfig):
        super().__init__()
        # Encoder Block Initializations 
        self.cfg1 = cfg1
        self.tok_emb1 = nn.Embedding(cfg1.vocab_size, cfg1.d_model)
        if cfg1.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg1.max_seq_len, cfg1.d_model)
            self.register_buffer("pos_emb_enc", pe, persistent=False)
   
        self.encblocks = nn.ModuleList([EncBlock(cfg1) for _ in range(cfg1.n_layers)])
        self.ln_f_enc = nn.LayerNorm(cfg1.d_model)
        # Decoder Block Initializations
        self.cfg2 = cfg2
        self.tok_emb2 = nn.Embedding(cfg2.vocab_size, cfg2.d_model)
        if cfg2.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg2.max_seq_len, cfg2.d_model)
            self.register_buffer("pos_emb_dec", pe, persistent=False)
       
        self.decblocks = nn.ModuleList([DecBlock(cfg2) for _ in range(cfg2.n_layers)])
        self.ln_f_dec = nn.LayerNorm(cfg2.d_model)
        self.head = nn.Linear(cfg2.d_model, cfg2.vocab_size, bias=False)
        self.head.weight = self.tok_emb2.weight
    def encode(self,src:torch.Tensor)->torch.Tensor:
        # passing it through encoder 
        B,S = src.shape
        
        x_enc = self.tok_emb1(src)
        if(self.cfg1.pos_emb_type == "sinusoidal"):
            x_enc=x_enc+self.pos_emb_enc[:S]
        for block in self.encblocks:
            x_enc = block(x_enc)
        x_enc = self.ln_f_enc(x_enc)
        memory = x_enc
        return memory

    def decode(self,tgt:torch.Tensor,memory:torch.Tensor)->torch.Tensor:
        B,T = tgt.shape 
        x_dec = self.tok_emb2(tgt)
        if(self.cfg2.pos_emb_type == "sinusoidal"):
            x_dec=x_dec+self.pos_emb_dec[:T]
        for block in self.decblocks:
            x_dec = block(x_dec,memory)
        x_dec = self.ln_f_dec(x_dec)
        x_dec = self.head(x_dec)
        return x_dec


    def forward(self ,src:torch.Tensor , tgt:torch.Tensor)->torch.Tensor:
        return self.decode(tgt,self.encode(src))
        
         
        






        
    
        

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/prefix_attention.py
# your prefix_attention.py (verbatim)
""" Prefix Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        
        self.register_buffer("causal_mask", mask, persistent=False)

        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
     

    def forward(self, x , p):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q,self.rope_cos[:T],self.rope_sin[:T])
            k = apply_rope(k,self.rope_cos[:T],self.rope_sin[:T])
        

        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)

        # TODO: same attention as before, but compute scores as
        #       (q @ k.transpose(-2, -1)) * self.attention_scale before masking/softmax
        attention_scores = (q@k.transpose(-2,-1))*self.attention_scale

        mask1 = self.causal_mask[:T,:T].clone()
        mask1[:p,:p]=1

        attention_scores = attention_scores.masked_fill(
            ~mask1,float("-inf")
        )
       
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.attn_dropout(attention_weights)
    
        weighted_sum = attention_weights@v
   
        weighted_sum = torch.reshape(weighted_sum.transpose(1,2),(B,T,C))
        output = self.out_proj(weighted_sum)
        output = self.resid_dropout(output)
        
        return output

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/prefix_lm.py
# your prefix_lm.py (verbatim)
"""Prefix LM - Decoder-only Transformer LM -- Block now passes use_rope through to attention, and
DecoderLM only builds/adds the sinusoidal table when pos_emb_type == "sinusoidal"
(everything else unchanged from Question 2). Different Architecture as mask changes the way information flows"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
from prefix_attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook
    norm_first: bool = False
    prefix:int = 128


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    PE = torch.zeros(max_len, d_model)

    for pos in range(max_len):
        PE[pos, 0::2] = torch.sin(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
        PE[pos, 1::2] = torch.cos(
        pos / (10000 ** (torch.arange(0, d_model, 2) / d_model))
    )
    return PE  # unchanged from Question 1


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.GELU(),
            nn.Linear(d_ff,d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg=cfg
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x,p):
        if self.cfg.norm_first is True:
            y1 = self.ln1(x)
            y2 = self.attn(y1,p)+x
            y3 = self.ln2(y2)
            y4 = self.ffn(y3)+y2
            output=y4
        else:
            y1 = self.attn(x,p)
            y2 = self.ln1(y1+x)
            y3 = self.ffn(y2)
            y4 = self.ln2(y3+y2)
            output=y4
            
        
        # and post-norm when norm_first=False.
        return output


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        # TODO: x = token embedding + positional encoding (if sinusoidal)
        x = self.tok_emb(idx)
        if(self.cfg.pos_emb_type == "sinusoidal"):
            x=x+self.pos_emb[:T]        # TODO: run x through every block, then ln_f, then self.head
        for block in self.blocks:
            x = block(x,self.cfg.prefix)
        x = self.ln_f(x)
        x = self.head(x)
        
        return x  # unchanged from Question 1 (just remember: only add
        # self.pos_emb when self.cfg.pos_emb_type == "sinusoidal")

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/fastattn_ab.py
# Plumbing: fused SDPA attention + KV cache + generation around your modules
"""Efficiency plumbing around the hand-written A3 models (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay
exactly as defined in arch1/arch2/arch3. `patch_model` swaps each attention module's forward
for one that computes the *same* function with fused kernels, and adds an optional KV cache
for generation. `check_equivalence` in the notebook verifies outputs match the original code.

* Training: F.scaled_dot_product_attention(is_causal=True). On T4 (sm_75) PyTorch cannot use
  the FlashAttention-2 kernel (needs sm_80+); it dispatches to the memory-efficient fused
  kernel, which gives the same benefit: no (T x T) score matrix materialised in HBM.
* Generation: per-layer KV cache (keys/values for MHA/GQA, the compressed latent for MLA),
  prefill once, then one token per step.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forward (drop-in for all three hand-written attention classes)
# ---------------------------------------------------------------------------

def _sdpa(q, k, v, causal, dropout_p, scale):
    return F.scaled_dot_product_attention(q, k, v, is_causal=causal, dropout_p=dropout_p, scale=scale)


def _fast_attention_forward(self, x):
    B, T, C = x.shape
    cache = getattr(self, "_kv_cache", None)
    start = cache["pos"] if cache is not None else 0
    n_heads, d_head = self.n_heads, self.d_head
    dropout_p = self.attn_dropout.p if self.training else 0.0

    q = self.q_proj(x).view(B, T, n_heads, d_head).transpose(1, 2)
    n_prefix = 0                                            # prefix tuning is only wired for MHA / GQA

    if hasattr(self, "kv_down_proj"):                       # MLA: cache the shared latent
        latent = self.kv_down_proj(x)
        if cache is not None:
            cache["latent"][:, start:start + T] = latent
            latent = cache["latent"][:, :start + T]
        S = latent.shape[1]
        k = self.k_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
        v = self.v_up_proj(latent).view(B, S, n_heads, d_head).transpose(1, 2)
    else:                                                   # MHA / GQA (+ optional RoPE)
        n_kv = getattr(self, "n_kv_heads", n_heads)
        k = self.k_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, n_kv, d_head).transpose(1, 2)
        if getattr(self, "use_rope", False):
            cos = self.rope_cos[start:start + T]
            sin = self.rope_sin[start:start + T]
            q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
            k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
        if cache is not None:
            cache["k"][:, :, start:start + T] = k
            cache["v"][:, :, start:start + T] = v
            k = cache["k"][:, :, :start + T]
            v = cache["v"][:, :, :start + T]
        k, v, n_prefix = _add_prefix(self, k, v, B, cache)
        n_rep = n_heads // n_kv
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

    # prefill (start == 0) is causal; a single decode step attends to the whole cache
    causal = T > 1
    if causal and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    if n_prefix and T > 1:                                  # prefix tuning: every query also sees the P prefix keys
        mask = _causal_with_prefix(T, n_prefix, x.device)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dropout_p, scale=self.attention_scale)
    else:
        y = _sdpa(q, k, v, causal, dropout_p, self.attention_scale)
    y = y.transpose(1, 2).reshape(B, T, n_heads * d_head)
    return self.resid_dropout(self.out_proj(y))


# ---------------------------------------------------------------------------
# prefix-tuning hook (Phase 3 plumbing): an attention module may carry `prefix_kv`, a callable
# prefix_kv(batch_size) -> (pk, pv), each (B, n_kv_heads, P, d_head). Those P key/value vectors are prepended to the
# keys/values (no RoPE is applied to them) and every query may attend to them. Without `prefix_kv`: unchanged.
# ---------------------------------------------------------------------------

def _add_prefix(attn, k, v, B, cache):
    fn = getattr(attn, "prefix_kv", None)
    if fn is None:
        return k, v, 0
    # Phase 6: also used with the KV cache (prefix vectors are re-attached each step; they are not cached)
    pk, pv = fn(B)
    assert pk.shape[0] == B and pk.shape[1] == k.shape[1] and pk.shape[3] == k.shape[3], \
        f"prefix_kv must return (B, n_kv_heads, P, d_head) = ({B}, {k.shape[1]}, P, {k.shape[3]}), got {tuple(pk.shape)}"
    return torch.cat([pk.to(k.dtype), k], dim=2), torch.cat([pv.to(v.dtype), v], dim=2), pk.shape[2]


def _causal_with_prefix(T, P, device):
    m = torch.ones(T, P + T, dtype=torch.bool, device=device)
    m[:, P:] = torch.ones(T, T, dtype=torch.bool, device=device).tril()
    return m


def core_attention(m):
    """Phase 6: the patched attention module itself, also when a technique wrapped block.attn (e.g. adapters)."""
    for sub in m.modules():
        if "_kv_cache" in sub.__dict__ or hasattr(sub, "_kv_cache"):
            return sub
    return m


def attention_modules(model):
    return [core_attention(block.attn) for block in model.blocks]


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    for attn in [block.attn for block in model.blocks]:
        attn._kv_cache = None
        if getattr(attn, "use_rope", False):
            # the user's own apply_rope, from the module that defines this attention class
            attn._apply_rope = sys.modules[type(attn).__module__].apply_rope
        attn.forward = types.MethodType(_fast_attention_forward, attn)
    return model


# ---------------------------------------------------------------------------
# KV cache
# ---------------------------------------------------------------------------

@contextmanager
def kv_cache(model, batch_size, max_len, dtype):
    """Allocate a per-layer cache. Inside the context call `step(model, ids)` repeatedly."""
    dev = next(model.parameters()).device
    caches = []
    for attn in attention_modules(model):
        c = {"pos": 0}
        if hasattr(attn, "kv_down_proj"):
            c["latent"] = torch.empty(batch_size, max_len, attn.kv_latent_dim, device=dev, dtype=dtype)
        else:
            n_kv = getattr(attn, "n_kv_heads", attn.n_heads)
            shape = (batch_size, n_kv, max_len, attn.d_head)
            c["k"] = torch.empty(shape, device=dev, dtype=dtype)
            c["v"] = torch.empty(shape, device=dev, dtype=dtype)
        attn._kv_cache = c
        caches.append(c)
    full_pe = getattr(model, "pos_emb", None)
    model._fast_state = {"caches": caches, "full_pe": full_pe}
    try:
        yield
    finally:
        for attn in attention_modules(model):
            attn._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe
        del model._fast_state


def step(model, ids):
    """Run `ids` (B, T) through the model at the current cache position; returns logits (B, T, V)."""
    state = model._fast_state
    pos = state["caches"][0]["pos"]
    if state["full_pe"] is not None:
        # DecoderLM.forward adds self.pos_emb[:T]; shift the table so rows pos..pos+T are used
        model.pos_emb = state["full_pe"][pos:]
    logits = model(ids)
    for c in state["caches"]:
        c["pos"] += ids.shape[1]
    return logits


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p          # keep the smallest set with mass >= top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def ngram_ban(seq, n, vocab):
    """(B, V) bool: tokens that would complete an n-gram (in token ids) already present in `seq` (B, S).
    For each row, every earlier occurrence of the last n-1 tokens bans the token that followed it."""
    B, S = seq.shape
    if n < 2 or S < n:
        return torch.zeros(B, vocab, dtype=torch.bool, device=seq.device)
    grams = seq.unfold(1, n, 1)                                   # (B, S-n+1, n)
    hit = (grams[:, :, :-1] == seq[:, None, S - n + 1:]).all(-1)  # earlier (n-1)-gram == current suffix
    cnt = torch.zeros(B, vocab, device=seq.device).scatter_add_(1, grams[:, :, -1], hit.float())
    return cnt > 0


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens,
                   batch_size, temperature, top_k, top_p, repetition_penalty, device, amp_dtype,
                   log_every=10, no_repeat_ngram=0, ngram_scope="output"):
    """no_repeat_ngram = n > 1 forbids repeating any n-gram of TOKEN ids (HF no_repeat_ngram_size), counted over
    the generated text only (ngram_scope="output") or over prompt + generated text ("all")."""
    assert ngram_scope in ("output", "all"), ngram_scope
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    Prompts are bucketed by token length so a batch needs no padding: every prompt in a batch
    keeps its last L tokens, L = shortest prompt in the batch (capped so prompt + new tokens fit
    max_seq_len). Words are counted from the tokenizer's end-of-word flag, so we stop as soon as
    every row has produced target_words complete words.
    """
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    assert max_prompt > 0, "max_new_tokens must be smaller than max_seq_len"
    results = [None] * len(prompts)

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, B, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            history = ids
            for t in range(max_new_tokens):
                # a bare </w> right after a finished word never occurs in encoded text: forbid it
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = history if ngram_scope == "all" else (torch.stack(out, 1) if out else ids[:, :0])
                    ban = ngram_ban(seq, no_repeat_ngram, logits.size(1))
                    ban &= ~(ban | torch.isinf(logits)).all(-1, keepdim=True)  # never leave a row with no option
                    logits = logits.masked_fill(ban, -float("inf"))
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                # a word ends on an end-of-word token that closes something non-empty
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()):
                    break
                if repetition_penalty != 1.0 or (no_repeat_ngram and ngram_scope == "all"):
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(model, nxt[:, None])[:, -1]
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


@torch.inference_mode()
def generate_beam(model, tokenizer, prompts, *, max_seq_len, target_words, max_new_tokens, batch_size,
                  beam_size, device, amp_dtype, no_repeat_ngram=0, ngram_scope="output", length_penalty=1.0,
                  log_every=10):
    """Beam search with the same word budget / token constraints as generate_words.
    Beams are ranked by sum(log p) / length**length_penalty (1.0 = mean log-prob per token). A beam that has
    produced `target_words` words is frozen (score and length fixed) and keeps competing as a finished
    hypothesis. Optional n-gram blocking per beam (same rule as generate_words). Returns list[str] in input order."""
    assert ngram_scope in ("output", "all"), ngram_scope
    model.eval()
    W = int(beam_size)
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    n_decodable = len(tokenizer.token_bytes)
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    max_prompt = max_seq_len - max_new_tokens
    per_batch = max(1, batch_size // W)
    results = [None] * len(prompts)
    neg_inf = -float("inf")

    for b, start in enumerate(range(0, len(order), per_batch)):
        idxs = order[start:start + per_batch]
        P = len(idxs)
        R = P * W
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        prompt_ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        ids = prompt_ids.repeat_interleave(W, 0)                                   # (R, L): row = p * W + beam
        words = torch.zeros(R, dtype=torch.int32, device=device)
        in_word = torch.zeros(R, dtype=torch.bool, device=device)
        done = torch.zeros(R, dtype=torch.bool, device=device)
        cum = torch.zeros(R, device=device)
        length = torch.zeros(R, device=device)
        gen = torch.zeros(R, 0, dtype=torch.long, device=device)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), \
                kv_cache(model, R, L + max_new_tokens, amp_dtype if device.type == "cuda" else torch.float32):
            logits = step(model, ids)[:, -1]
            for t in range(max_new_tokens):
                lp = torch.nan_to_num(logits.float(), nan=-1e4)
                lp[:, :tokenizer.n_special] = neg_inf
                lp[~in_word, tokenizer.eow_id] = neg_inf
                lp[:, n_decodable:] = neg_inf
                if no_repeat_ngram and no_repeat_ngram > 1:
                    seq = gen if ngram_scope == "output" else torch.cat([ids, gen], 1)
                    ban = ngram_ban(seq, no_repeat_ngram, lp.size(1))
                    ban &= ~(ban | torch.isinf(lp)).all(-1, keepdim=True)
                    lp = lp.masked_fill(ban, neg_inf)
                lp = lp.log_softmax(-1)
                V = lp.size(1)
                # candidate scores (length-normalised); finished beams keep their score via one frozen continuation
                cand = (cum[:, None] + lp) / (length[:, None] + 1).pow(length_penalty)
                frozen_tok = lp.argmax(-1)
                frozen = torch.full_like(cand, neg_inf)
                frozen.scatter_(1, frozen_tok[:, None], (cum / length.clamp(min=1).pow(length_penalty))[:, None])
                cand = torch.where(done[:, None], frozen, cand)
                if t == 0:                                                         # identical beams: expand beam 0 only
                    cand.view(P, W, V)[:, 1:] = neg_inf
                top_s, top_i = cand.view(P, W * V).topk(W, dim=-1)                 # (P, W)
                parent = (top_i // V) + torch.arange(P, device=device)[:, None] * W
                parent, tok = parent.reshape(-1), (top_i % V).reshape(-1)
                was_done = done[parent]
                # reorder beam state + KV cache to the chosen parents
                for c in model._fast_state["caches"]:
                    pos = c["pos"]
                    c["k"][:, :, :pos] = c["k"][parent, :, :pos]
                    c["v"][:, :, :pos] = c["v"][parent, :, :pos]
                words, in_word, done = words[parent], in_word[parent], done[parent].clone()
                cum = torch.where(was_done, cum[parent], cum[parent] + lp[parent, tok])
                length = torch.where(was_done, length[parent], length[parent] + 1)
                gen = torch.cat([gen[parent], tok[:, None]], 1)
                open_ = in_word | has_bytes[tok]
                words = words + (eow[tok] & open_ & ~was_done).int()
                in_word = open_ & ~eow[tok]
                done = was_done | (words >= target_words)
                if bool(done.all()):
                    break
                logits = step(model, tok[:, None])[:, -1]
        score = (cum / length.clamp(min=1).pow(length_penalty)).view(P, W)
        score = torch.where(done.view(P, W), score, score - 1e6)                     # prefer finished beams
        best = score.argmax(-1) + torch.arange(P, device=device) * W
        rows = gen[best].tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(rows[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  beam generated {min(start + per_batch, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence(make_model, vocab_size, seq_len=48, rtol=1e-4):
    """Original forward vs patched forward vs KV-cached incremental decoding, fp32 on CPU.
    Returns max |difference| relative to the largest |logit| of the original model."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    with kv_cache(fast, 2, seq_len, torch.float32):
        pre = step(fast, idx[:, :seq_len // 2])
        steps = [step(fast, idx[:, t:t + 1]) for t in range(seq_len // 2, seq_len)]
    err_cache = (a - torch.cat([pre] + steps, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile /kaggle/working/phase6/phase1_models/fastattn_cd.py
# Plumbing: fused SDPA attention + KV caches + generation around your #3 / #4 modules
"""Efficiency plumbing around the hand-written encoder-decoder (#3) and prefix-LM (#4) (written with Claude; declare it).

Nothing here changes an architecture: parameters, shapes and the maths of every model stay exactly as defined in
your files. `patch_model` swaps each attention module's forward for one that computes the *same* function with
F.scaled_dot_product_attention (memory-efficient fused kernel on T4), and adds optional caches for generation:

  encoder SelfAttention (non_causal_self_attention.py)  -> bidirectional, no cache (the encoder runs once per prompt)
  decoder CausalSelfAttention (attention3.py)           -> causal, self-attention KV cache
  CrossAttention (cross_attention.py)                   -> K/V of the encoder memory computed once per prompt, cached
  prefix CausalSelfAttention (prefix_attention.py)      -> mask = causal OR [:p, :p] block, KV cache for decoding

`check_equivalence_*` (run in the notebook before training) verify the patched model against your original forward,
and cached step-by-step decoding against a full forward pass.
"""
import sys
import types
from contextlib import contextmanager

import torch
import torch.nn.functional as F


# ---------------------------------------------------------------------------
# attention forwards
# ---------------------------------------------------------------------------

def _dropout_p(self):
    return self.attn_dropout.p if self.training else 0.0


def _project_self(self, x, start):
    """q, k, v of a self-attention module, RoPE at absolute positions start..start+T (as in your code)."""
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
    if getattr(self, "use_rope", False):
        cos = self.rope_cos[start:start + T]
        sin = self.rope_sin[start:start + T]
        q = self._apply_rope(q.float(), cos, sin).to(v.dtype)
        k = self._apply_rope(k.float(), cos, sin).to(v.dtype)
    return q, k, v


def _use_cache(self, k, v, start):
    cache = getattr(self, "_kv_cache", None)
    if cache is None:
        return k, v
    T = k.shape[2]
    cache["k"][:, :, start:start + T] = k
    cache["v"][:, :, start:start + T] = v
    return cache["k"][:, :, :start + T], cache["v"][:, :, :start + T]


def _expand(self, k, v):
    if self.n_rep > 1:
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
    return k, v


def _finish(self, y):
    B, H, T, D = y.shape
    return self.resid_dropout(self.out_proj(y.transpose(1, 2).reshape(B, T, H * D)))


def _sdpa(self, q, k, v, attn_mask=None, is_causal=False):
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal,
                                          dropout_p=_dropout_p(self), scale=self.attention_scale)


def _start(self):
    cache = getattr(self, "_kv_cache", None)
    return cache["pos"] if cache is not None else 0


def _prefix_hook(self, k, v, B):
    """prefix-tuning hook (Phase 3 plumbing): optional self.prefix_kv(B) -> (pk, pv), each (B, n_kv_heads, P, d_head),
    prepended to keys/values (no RoPE on them). Returns (k, v, P)."""
    fn = getattr(self, "prefix_kv", None)
    if fn is None:
        return k, v, 0
    # Phase 6: also used with the KV cache (prefix vectors are re-attached each step; they are not cached)
    pk, pv = fn(B)
    assert pk.shape[0] == B and pk.shape[1] == k.shape[1] and pk.shape[3] == k.shape[3], \
        f"prefix_kv must return (B, n_kv_heads, P, d_head) = ({B}, {k.shape[1]}, P, {k.shape[3]}), got {tuple(pk.shape)}"
    return torch.cat([pk.to(k.dtype), k], dim=2), torch.cat([pv.to(v.dtype), v], dim=2), pk.shape[2]


def _with_prefix_cols(mask_TT, P):
    """(T, T) bool mask -> (T, P + T): the P prefix columns are visible to every query."""
    T = mask_TT.shape[0]
    return torch.cat([torch.ones(T, P, dtype=torch.bool, device=mask_TT.device), mask_TT], dim=1)


def _fwd_bidirectional(self, x):
    """encoder SelfAttention: no mask."""
    q, k, v = _project_self(self, x, 0)
    k, v, P = _prefix_hook(self, k, v, x.shape[0])
    k, v = _expand(self, k, v)
    return _finish(self, _sdpa(self, q, k, v))


def _fwd_causal(self, x):
    """decoder CausalSelfAttention: causal; one-token steps attend to the whole cache."""
    start = _start(self)
    q, k, v = _project_self(self, x, start)
    k, v = _use_cache(self, k, v, start)
    k, v, P = _prefix_hook(self, k, v, x.shape[0])
    k, v = _expand(self, k, v)
    T = x.shape[1]
    if T > 1 and start > 0:
        raise ValueError("multi-token steps are only supported from an empty cache")
    if P and T > 1:
        mask = _with_prefix_cols(torch.ones(T, T, dtype=torch.bool, device=x.device).tril(), P)
        return _finish(self, _sdpa(self, q, k, v, attn_mask=mask))
    return _finish(self, _sdpa(self, q, k, v, is_causal=T > 1 and not P))


def _prefix_mask(self, T, p, device):
    key = (T, int(p), str(device))
    masks = self.__dict__.setdefault("_mask_cache", {})
    if key not in masks:
        m = torch.ones(T, T, dtype=torch.bool, device=device).tril()
        m[:p, :p] = True                       # same rule as your prefix_attention.py
        masks[key] = m
    return masks[key]


def _fwd_prefix(self, x, p):
    """prefix attention: causal OR [:p, :p]; one-token decoding steps attend to the whole cache."""
    start = _start(self)
    q, k, v = _project_self(self, x, start)
    k, v = _use_cache(self, k, v, start)
    k, v, P = _prefix_hook(self, k, v, x.shape[0])
    k, v = _expand(self, k, v)
    T = x.shape[1]
    if start == 0:
        mask = _prefix_mask(self, T, p, x.device)
        if P:
            mask = _with_prefix_cols(mask, P)
        return _finish(self, _sdpa(self, q, k, v, attn_mask=mask))
    if T == 1:
        return _finish(self, _sdpa(self, q, k, v))
    raise ValueError("multi-token steps are only supported from an empty cache")


def _fwd_cross(self, x, memory):
    """CrossAttention: q from the decoder state, k/v from the encoder memory (cached once per prompt)."""
    assert not getattr(self, "use_rope", False), "RoPE in cross-attention is not supported (your design: use_rope=False)"
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
    cache = getattr(self, "_kv_cache", None)
    if cache is not None and "k" in cache:
        k, v = cache["k"], cache["v"]
    else:
        S = memory.shape[1]
        k = self.k_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(memory).view(B, S, self.n_kv_heads, self.d_head).transpose(1, 2)
        if cache is not None:
            cache["k"], cache["v"] = k, v
    k, v = _expand(self, k, v)
    return _finish(self, _sdpa(self, q, k, v))


_FORWARDS = {
    ("non_causal_self_attention", "SelfAttention"): _fwd_bidirectional,
    ("attention3", "CausalSelfAttention"): _fwd_causal,
    ("cross_attention", "CrossAttention"): _fwd_cross,
    ("prefix_attention", "CausalSelfAttention"): _fwd_prefix,
}


def patch_model(model):
    """Swap every attention forward for the fused/cached version (same parameters)."""
    n = 0
    for mod in model.modules():
        fwd = _FORWARDS.get((type(mod).__module__, type(mod).__name__))
        if fwd is None:
            continue
        mod._kv_cache = None
        if getattr(mod, "use_rope", False):
            mod._apply_rope = sys.modules[type(mod).__module__].apply_rope   # your own apply_rope
        mod.forward = types.MethodType(fwd, mod)
        n += 1
    assert n > 0, "no attention module recognised; did a module/class name change?"
    return model


# ---------------------------------------------------------------------------
# generation sessions: prefill once, then one token per step
# ---------------------------------------------------------------------------

def _core(m):
    """Phase 6: the patched attention module itself, also when a technique wrapped it (e.g. adapters)."""
    for sub in m.modules():
        if hasattr(sub, "_kv_cache"):
            return sub
    return m


def _alloc_self_cache(attn, B, max_len, dtype, device):
    attn = _core(attn)
    shape = (B, attn.n_kv_heads, max_len, attn.d_head)
    attn._kv_cache = {"pos": 0, "k": torch.empty(shape, device=device, dtype=dtype),
                      "v": torch.empty(shape, device=device, dtype=dtype)}
    return attn._kv_cache


@contextmanager
def prefix_session(model, ids, max_new_tokens, dtype):
    """#4: prefill the prompt (mask uses the model's own cfg.prefix), then decode with the KV cache."""
    B, L = ids.shape
    dev = ids.device
    caches = [_alloc_self_cache(b.attn, B, L + max_new_tokens, dtype, dev) for b in model.blocks]
    full_pe = getattr(model, "pos_emb", None)

    def run(tok):
        pos = caches[0]["pos"]
        if full_pe is not None:
            model.pos_emb = full_pe[pos:]     # DecoderLM adds pos_emb[:T]; shift it to rows pos..pos+T
        logits = model(tok)
        for c in caches:
            c["pos"] += tok.shape[1]
        return logits[:, -1]

    try:
        yield run(ids), run
    finally:
        for b in model.blocks:
            _core(b.attn)._kv_cache = None
        if full_pe is not None:
            model.pos_emb = full_pe


@contextmanager
def encdec_session(model, ids, max_new_tokens, dtype, start_id=None):
    """#3: encode the prompt once, cache every decoder layer's cross-attention K/V, decode token by token.
    start_id=None -> the decoder starts from the last source token, else from start_id (e.g. <s>)."""
    B = ids.shape[0]
    dev = ids.device
    memory = model.encode(ids)
    caches = [_alloc_self_cache(b.attn, B, max_new_tokens + 1, dtype, dev) for b in model.decblocks]
    for b in model.decblocks:
        _core(b.crattn)._kv_cache = {}
    full_pe = getattr(model, "pos_emb_dec", None)

    def run(tok):
        pos = caches[0]["pos"]
        if full_pe is not None:
            model.pos_emb_dec = full_pe[pos:]
        logits = model.decode(tok, memory)
        for c in caches:
            c["pos"] += tok.shape[1]
        return logits[:, -1]

    first = ids[:, -1:] if start_id is None else torch.full((B, 1), start_id, device=dev, dtype=ids.dtype)
    try:
        yield run(first), run
    finally:
        for b in model.decblocks:
            _core(b.attn)._kv_cache = None
            _core(b.crattn)._kv_cache = None
        if full_pe is not None:
            model.pos_emb_dec = full_pe


# ---------------------------------------------------------------------------
# sampling + word-budgeted generation
# ---------------------------------------------------------------------------

def sample_next(logits, temperature, top_k, top_p, banned_ids, prev_ids=None, repetition_penalty=1.0):
    logits = torch.nan_to_num(logits.float(), nan=-1e4)   # a diverged checkpoint must not crash sampling
    if banned_ids is not None:
        logits[:, banned_ids] = -float("inf")
    if repetition_penalty != 1.0 and prev_ids is not None:
        seen = torch.gather(logits, 1, prev_ids)
        seen = torch.where(seen > 0, seen / repetition_penalty, seen * repetition_penalty)
        logits = logits.scatter(1, prev_ids, seen)
    if not temperature:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k:
        kth = torch.topk(logits, top_k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p and top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        drop = probs.cumsum(-1) - probs > top_p
        sorted_logits = sorted_logits.masked_fill(drop, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


@torch.inference_mode()
def generate_words(model, tokenizer, prompts, *, kind, max_prompt, target_words, max_new_tokens, batch_size,
                   temperature, top_k, top_p, repetition_penalty, device, amp_dtype, start_id=None, log_every=10):
    """Continue each prompt by `target_words` whitespace words. Returns list[str] in input order.

    kind = "prefix" (#4) or "encdec" (#3). Prompts are bucketed by token length so a batch needs no padding: each
    prompt keeps its last L tokens, L = shortest prompt in the batch (capped at max_prompt)."""
    model.eval()
    eow = torch.tensor(tokenizer.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tokenizer.token_bytes], device=device, dtype=torch.bool)
    banned = torch.arange(tokenizer.n_special, device=device)     # never emit pad/unk/</s>/...
    encoded = [tokenizer.encode(p) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: len(encoded[i]))
    results = [None] * len(prompts)
    cache_dtype = amp_dtype if device.type == "cuda" else torch.float32

    for b, start in enumerate(range(0, len(order), batch_size)):
        idxs = order[start:start + batch_size]
        L = min(min(len(encoded[i]) for i in idxs), max_prompt)
        ids = torch.tensor([encoded[i][-L:] for i in idxs], device=device, dtype=torch.long)
        B = len(idxs)
        words = torch.zeros(B, dtype=torch.int32, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)     # prompts end on a full word
        done = torch.zeros(B, dtype=torch.bool, device=device)
        out = []
        session = (prefix_session(model, ids, max_new_tokens, cache_dtype) if kind == "prefix"
                   else encdec_session(model, ids, max_new_tokens, cache_dtype, start_id))
        with torch.autocast("cuda", dtype=amp_dtype, enabled=device.type == "cuda"), session as (logits, step):
            history = ids
            for t in range(max_new_tokens):
                logits = logits.float().clone()
                logits[~in_word, tokenizer.eow_id] = -float("inf")       # a bare </w> after a finished word
                logits[:, len(tokenizer.token_bytes):] = -float("inf")   # ids the tokenizer can't decode
                nxt = sample_next(logits, temperature, top_k, top_p, banned,
                                  history if repetition_penalty != 1.0 else None, repetition_penalty)
                out.append(nxt)
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done).int()
                in_word = open_ & ~eow[nxt]
                done |= words >= target_words
                if bool(done.all()) or t == max_new_tokens - 1:
                    break
                if repetition_penalty != 1.0:
                    history = torch.cat([history, nxt[:, None]], 1)
                logits = step(nxt[:, None])
        gen = torch.stack(out, 1).tolist()
        for row, i in enumerate(idxs):
            text = tokenizer.decode(gen[row])
            results[i] = " ".join(text.split()[:target_words])
        if log_every and (b + 1) % log_every == 0:
            print(f"  generated {min(start + batch_size, len(order))}/{len(order)}", flush=True)
    return results


# ---------------------------------------------------------------------------
# verification helpers (run in the notebook before training)
# ---------------------------------------------------------------------------

@torch.no_grad()
def check_equivalence_encdec(make_model, vocab_size, src_len=20, tgt_len=12, rtol=1e-4):
    """Your forward vs patched forward vs the generation path (encdec_session: encode once, cached cross-attention
    K/V, one decoder token per step). Returns relative max errors (fp32, CPU)."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    src = torch.randint(0, vocab_size, (2, src_len))
    tgt = torch.randint(0, vocab_size, (2, tgt_len))
    tgt[:, 0] = src[:, -1]                    # session with start_id=None starts from the last source token
    a = ref(src, tgt)
    scale = a.abs().max().item()
    err_full = (a - fast(src, tgt)).abs().max().item() / scale
    with encdec_session(fast, src, tgt_len, torch.float32, start_id=None) as (first, step):
        outs = [first[:, None]]
        for t in range(1, tgt_len):
            outs.append(step(tgt[:, t:t + 1])[:, None])
    err_cache = (a - torch.cat(outs, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache


@torch.no_grad()
def check_equivalence_prefix(make_model, vocab_size, prefix, seq_len=None, rtol=1e-4):
    """Your forward vs patched forward vs (prefill >= prefix tokens, then cached one-token steps)."""
    torch.manual_seed(0)
    ref = make_model().eval()
    fast = make_model().eval()
    fast.load_state_dict(ref.state_dict())
    patch_model(fast)
    seq_len = seq_len or prefix + 12
    idx = torch.randint(0, vocab_size, (2, seq_len))
    a = ref(idx)
    scale = a.abs().max().item()
    err_full = (a - fast(idx)).abs().max().item() / scale
    n_new = seq_len - prefix
    with prefix_session(fast, idx[:, :prefix], n_new, torch.float32) as (first, step):
        outs = [first[:, None]]
        for t in range(prefix, seq_len - 1):
            outs.append(step(idx[:, t:t + 1])[:, None])
    err_cache = (a[:, prefix - 1:seq_len - 1] - torch.cat(outs, 1)).abs().max().item() / scale
    assert err_full < rtol and err_cache < rtol, (err_full, err_cache)
    return err_full, err_cache

In [ ]:
%%writefile /kaggle/working/phase6/fetch_data.py
#!/usr/bin/env python3
"""Download Phase_6_Summ/ (plumbing; declare it).

Token lookup order: Kaggle secret "HF" -> HF_TOKEN environment variable -> hidden prompt.
The token is used for these requests only: never printed, never written to disk.

  python fetch_data.py                 # -> ./data/Phase_6_Summ/{Train/train.json,Test/test.json}
"""
import argparse
import getpass
import json
import os
import sys
import urllib.error
import urllib.request

REPO = "https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone/resolve/main/"
FILES = ["Phase_6_Summ/Train/train.json", "Phase_6_Summ/Test/test.json"]


def get_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        pass
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    if not sys.stdin.isatty():
        raise SystemExit("No Hugging Face token: on Kaggle add a secret with the LABEL 'HF' (Add-ons -> Secrets) and tick "
                         "it for this notebook; locally set HF_TOKEN or run this script in a terminal.")
    return getpass.getpass("Hugging Face token (input hidden): ").strip()


def get(url, token, rng=None):
    h = {"Authorization": f"Bearer {token}"}
    if rng:
        h["Range"] = f"bytes={rng[0]}-{rng[1]}"
    req = urllib.request.Request(url, headers=h)
    try:
        with urllib.request.urlopen(req, timeout=300) as r:
            return r.read(), dict(r.headers)
    except urllib.error.HTTPError as e:
        hint = "check the token / dataset access" if e.code in (401, 403) else e.reason
        raise SystemExit(f"download failed for {url.split('/main/')[-1]}: HTTP {e.code} ({hint})")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "data"))
    args = ap.parse_args()
    token = get_token()
    try:
        for rel in FILES:
            dst = os.path.join(args.out, rel)
            if os.path.exists(dst):
                print("exists:", dst); continue
            data, _ = get(REPO + rel, token)
            json.loads(data)                                   # validate before saving
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            open(dst, "wb").write(data)
            print(f"saved {dst} ({len(data) / 1e6:.1f} MB)")
    finally:
        del token


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase6/summ_data.py
"""Phase-6 data plumbing (declare it): load / clean / split, sentence-level ByteBPE encoding (cached), head+tail
source selection, sentence-packed chunks (hierarchical stage 1), target truncation and per-kind batch collation.

Every source is exactly src_len tokens (short ones are LEFT-padded with <pad>), so a batch never mixes source lengths:
D's bidirectional prefix boundary and the generation prefill are identical for every row. Only documents shorter than
src_len are padded (about 2% of train at 896 tokens, fewer at smaller src_len).

Sequence formats (S = src_len, Tt = tgt_len, the target includes the end token when it is appended):
  decoder / prefix (A, B, D): [src S][sep] [tgt ... </s>] [pad ...]      labels = next token on the target part only
                              D: bidirectional over [src + sep] (prefix_len = S + 1), causal on the target
  encdec (C):                 encoder: [src S];  decoder in: [start] + tgt;  labels: tgt (+ </s>)
"""
import hashlib
import json
import os
import pickle
import random
import re

import numpy as np
import torch

SENT = re.compile(r"(?<=[.?!])\s+(?=[A-Z(\"'\[])")


def norm_hash(text):
    return hashlib.md5(" ".join(text.split()).encode()).hexdigest()


def load_json(path):
    d = json.load(open(path))
    if isinstance(d, dict):
        d = [{"id": k, **v} if isinstance(v, dict) else {"id": k, "document": v} for k, v in d.items()]
    return d


def load_splits(data_dir, val_docs, seed):
    """-> train, val, test (lists of {"id", "document", ["summary"]}), report. val = seeded hold-out from train.json."""
    train = load_json(os.path.join(data_dir, "Train", "train.json"))
    test = load_json(os.path.join(data_dir, "Test", "test.json"))
    rep = {"train_raw": len(train), "test": len(test)}
    train = [d for d in train if d.get("summary", "").strip() and d.get("document", "").strip()]
    rep["dropped_empty"] = rep["train_raw"] - len(train)
    seen, uniq = set(), []
    for d in train:
        h = norm_hash(d["document"])
        if h not in seen:
            seen.add(h); uniq.append(d)
    rep["dropped_duplicate_docs"] = len(train) - len(uniq)
    test_h = {norm_hash(d["document"]) for d in test}
    train = [d for d in uniq if norm_hash(d["document"]) not in test_h]
    rep["dropped_overlapping_test"] = len(uniq) - len(train)
    order = sorted(train, key=lambda d: d["id"])
    random.Random(seed).shuffle(order)
    val, train = order[:val_docs], order[val_docs:]
    rep.update(train=len(train), val=len(val))
    return train, val, test, rep


def split_sentences(text):
    text = " ".join(text.split())
    return [s for s in SENT.split(text) if s]


def encode_sentences(tok, docs, field, cache_path=None, workers=None):
    """-> {id: [ids of sentence 0, ids of sentence 1, ...]} for docs[field]; cached (pickle) by ids + text hash."""
    key = hashlib.md5("".join(d["id"] + norm_hash(d[field]) for d in docs).encode()).hexdigest()
    if cache_path and os.path.exists(cache_path):
        c = pickle.load(open(cache_path, "rb"))
        if c.get("key") == key:
            return c["enc"]
    sents = [split_sentences(d[field]) for d in docs]
    flat = [s for ss in sents for s in ss]
    ids = list(tok.encode_many(flat, workers=workers)) if len(flat) > 2000 else [tok.encode(s) for s in flat]
    enc, k = {}, 0
    for d, ss in zip(docs, sents):
        enc[d["id"]] = ids[k:k + len(ss)]
        k += len(ss)
    if cache_path:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        pickle.dump({"key": key, "enc": enc}, open(cache_path + ".tmp", "wb"))
        os.replace(cache_path + ".tmp", cache_path)
    return enc


# ---------------------------------------------------------------------------
# source selection / chunking / targets
# ---------------------------------------------------------------------------

def select_headtail(sents, S, head_frac):
    """Whole sentences from the start up to head_frac * S tokens, then whole sentences from the end up to the rest of
    the budget; leftover budget is filled by continuing the head (partial sentence). Document order is kept.
    A document that fits is returned whole. -> list of <= S ids."""
    flat_len = sum(map(len, sents))
    if flat_len <= S:
        return [t for s in sents for t in s]
    head_budget = int(round(head_frac * S))
    head, used, i = [], 0, 0
    while i < len(sents) and used + len(sents[i]) <= head_budget:
        head.append(sents[i]); used += len(sents[i]); i += 1
    tail, j = [], len(sents) - 1
    while j >= i and used + len(sents[j]) <= S:
        tail.append(sents[j]); used += len(sents[j]); j -= 1
    head_ids = [t for s in head for t in s]
    gap = S - used
    if gap > 0 and i <= j:                       # partial: continue the head into the unused middle
        mid = [t for s in sents[i:j + 1] for t in s]
        head_ids += mid[:gap]
    out = head_ids + [t for s in reversed(tail) for t in s]
    assert len(out) <= S
    return out


def pack_chunks(sents, S):
    """Greedy sentence packing into chunks of <= S tokens (a sentence longer than S is split). -> list of id lists."""
    chunks, cur = [], []
    for s in sents:
        while len(s) > S:
            if cur:
                chunks.append(cur); cur = []
            chunks.append(s[:S]); s = s[S:]
        if len(cur) + len(s) > S:
            chunks.append(cur); cur = []
        cur = cur + s
    if cur:
        chunks.append(cur)
    return chunks


def truncate_target(sents, budget, mode="sentence"):
    """-> (ids, truncated?) with len(ids) <= budget. mode "sentence": whole sentences only (the first sentence is cut
    if it alone is too long); "token": plain token cut."""
    flat = [t for s in sents for t in s]
    if len(flat) <= budget:
        return flat, False
    if mode == "token":
        return flat[:budget], True
    out = []
    for s in sents:
        if len(out) + len(s) > budget:
            break
        out += s
    return (out or flat[:budget]), True


def pad_left(ids, S, pad_id=0):
    return [pad_id] * (S - len(ids)) + list(ids)


# ---------------------------------------------------------------------------
# examples + collation
# ---------------------------------------------------------------------------

def build_examples(docs, src_enc, tgt_enc, fmt, sel, eos_id):
    """docs: list of {"id"}; src_enc / tgt_enc: {id: [sentence ids]} -> list of {"id", "src" (S ids), "tgt"}.
    fmt: src_len, tgt_len, eos_on_truncated, target_trunc; sel: strategy ("headtail"), head_frac."""
    S, Tt = fmt["src_len"], fmt["tgt_len"]
    out, n_trunc, n_short = [], 0, 0
    for d in docs:
        src = select_headtail(src_enc[d["id"]], S, sel["head_frac"])
        n_short += len(src) < S
        ex = {"id": d["id"], "src": pad_left(src, S)}
        if tgt_enc is not None:
            # room for </s> is always reserved; a truncated target gets </s> only if eos_on_truncated
            tgt, trunc = truncate_target(tgt_enc[d["id"]], Tt - 1, fmt.get("target_trunc", "sentence"))
            if (not trunc) or fmt["eos_on_truncated"]:
                tgt = tgt + [eos_id]
            n_trunc += trunc
            ex["tgt"] = tgt
        out.append(ex)
    stats = {"examples": len(out), "padded_sources": n_short}
    if tgt_enc is not None:
        stats.update(truncated_targets=n_trunc,
                     mean_tgt_tokens=round(float(np.mean([len(e["tgt"]) for e in out])), 1))
    return out, stats


def collate(examples, kind, sep_id=None, start_id=None, device="cpu"):
    """-> batch dict for GenModel.target_logits."""
    B = len(examples)
    S = len(examples[0]["src"])
    assert all(len(e["src"]) == S for e in examples)
    if kind == "encdec":
        L = max(len(e["tgt"]) for e in examples)
        dec_in = torch.zeros(B, L, dtype=torch.long)
        labels = torch.full((B, L), -100, dtype=torch.long)
        for r, e in enumerate(examples):
            first = e["src"][-1] if start_id is None else start_id
            t = e["tgt"]
            dec_in[r, :len(t)] = torch.tensor([first] + t[:-1])
            labels[r, :len(t)] = torch.tensor(t)
        return {"src": torch.tensor([e["src"] for e in examples]).to(device), "dec_in": dec_in.to(device),
                "labels": labels.to(device)}
    L = S + 1 + max(len(e["tgt"]) for e in examples) - 1      # last target token is never an input
    ids = torch.zeros(B, L, dtype=torch.long)
    labels = torch.full((B, L), -100, dtype=torch.long)
    for r, e in enumerate(examples):
        seq = e["src"] + [sep_id] + e["tgt"]
        ids[r, :len(seq) - 1] = torch.tensor(seq[:-1])
        labels[r, S:S + len(e["tgt"])] = torch.tensor(e["tgt"])   # position S (= sep) predicts tgt[0]
    b = {"ids": ids.to(device), "labels": labels.to(device)}
    if kind == "prefix":
        b["prefix_len"] = S + 1
    return b

In [ ]:
%%writefile /kaggle/working/phase6/gen_model.py
"""Phase-1 checkpoints -> generation models for Phase-6 summarization (plumbing; declare it).

Your Phase-1 model code is used unchanged (phase1_models/). Differences to the Phase 2-5 Backbone:
  * the LM head is KEPT, but moved out of the backbone: GenModel = .backbone (what your technique files receive) +
    .head (the Phase-1 nn.Linear, still tied to the token embedding). Technique target matching (e.g. "q_proj") can
    therefore never hit the head; a frozen embedding (freeze(backbone)) also freezes the tied head.
  * kind "encdec" (C) uses BOTH encoder and decoder (encdec_mode = "seq2seq"); kind "prefix" (D) gets its prefix
    length per call (= source + separator).
  * prompt tuning: backbone.n_virtual_tokens = P -> P placeholder ids are prepended to the decoder input (A/B/D) or the
    encoder input (C). For A/B/D the P output positions are dropped again; for C the encoder memory keeps them (the
    decoder may cross-attend to the soft prompt).
  * prefix tuning: the fastattn prefix_kv hook now also works with the generation KV cache.

Backbone options (cfg["backbone"]):
  truncate_layers       keep the first k blocks (A/B/D blocks; C: k encoder AND k decoder blocks) - KD students.
  dropout               override every nn.Dropout p; None = keep the checkpoint's value.
  max_seq_len_override  A/B/D: int; C: {"encoder": int, "decoder": int}. Rebuilds the model with longer positional
                        tables / masks (they are not stored in the checkpoint). Positions beyond the Phase-1 length
                        were never trained (extrapolation).
"""
import copy
import importlib
import os
import sys
from contextlib import contextmanager

import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath(__file__))
for _p in (HERE, os.path.join(HERE, "phase1_models")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import fastattn_ab   # noqa: E402
import fastattn_cd   # noqa: E402


def _clean_state(sd):
    out = {}
    for k, v in sd.items():
        for pre in ("_orig_mod.", "module."):
            if k.startswith(pre):
                k = k[len(pre):]
        out[k] = v
    return out


def build_lm(spec, override=None):
    spec = copy.deepcopy(spec)
    kind = spec.get("kind", "decoder")
    arch = importlib.import_module(spec["module"])
    if kind in ("decoder", "prefix"):
        if override:
            spec["model"]["max_seq_len"] = int(override)
        lm = arch.DecoderLM(arch.LMConfig(**spec["model"]))
    elif kind == "encdec":
        if override:
            spec["encoder"]["max_seq_len"] = int(override.get("encoder") or spec["encoder"]["max_seq_len"])
            spec["decoder"]["max_seq_len"] = int(override.get("decoder") or spec["decoder"]["max_seq_len"])
        lm = arch.EncDecLM(arch.EncLMConfig(**spec["encoder"]), arch.DecLMConfig(**spec["decoder"]))
    else:
        raise ValueError(f"unknown model kind {kind}")
    return lm, kind, spec


def load_phase1(path, override=None, load_weights=True):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    lm, kind, spec = build_lm(ck["config"], override)
    if load_weights:
        sd = _clean_state(ck["model"])
        if override:     # positional buffers / masks are rebuilt for the new length: skip shape-mismatched buffers
            own = lm.state_dict()
            bad = [k for k, v in sd.items() if k in own and own[k].shape != v.shape]
            params = dict(lm.named_parameters())
            assert not [k for k in bad if k in params], f"override changes parameter shapes: {bad}"
            sd = {k: v for k, v in sd.items() if k not in bad}
            missing, unexpected = lm.load_state_dict(sd, strict=False)
            assert not [k for k in missing if k in params] and not unexpected, (missing, unexpected)
        else:
            lm.load_state_dict(sd)
    info = dict(path=path, kind=kind, module=spec["module"], tag=spec.get("tag"), step=ck.get("step"),
                best_val=ck.get("best_val"))
    return lm, kind, spec, info


class GenBackbone(nn.Module):
    """Attributes for technique code (same names as Phases 2-5): .lm, .kind, .blocks, .d_model, .max_len,
    .n_virtual_tokens, .encdec_mode ("seq2seq" for C), get_input_embedding / set_input_embedding."""

    def __init__(self, lm, kind, truncate_layers=None, dropout=None):
        super().__init__()
        self.kind = kind
        self.lm = lm
        self.encdec_mode = "seq2seq" if kind == "encdec" else None
        if kind == "encdec":
            self.d_model = lm.cfg1.d_model
            self.max_len = lm.cfg1.max_seq_len          # encoder (source) limit
            self.max_tgt_len = lm.cfg2.max_seq_len      # decoder limit
        else:
            self.d_model = lm.cfg.d_model
            self.max_len = lm.cfg.max_seq_len
            self.max_tgt_len = None
        self.n_virtual_tokens = 0
        if truncate_layers:
            k = int(truncate_layers)
            if kind == "encdec":
                lm.encblocks = nn.ModuleList(list(lm.encblocks)[:k])
                lm.decblocks = nn.ModuleList(list(lm.decblocks)[:k])
            else:
                lm.blocks = nn.ModuleList(list(lm.blocks)[:k])
        if dropout is not None:
            for m in self.modules():
                if isinstance(m, nn.Dropout):
                    m.p = float(dropout)

    @property
    def blocks(self):
        if self.kind == "encdec":
            return list(self.lm.encblocks) + list(self.lm.decblocks)
        return list(self.lm.blocks)

    def get_input_embedding(self):
        return self.lm.tok_emb1 if self.kind == "encdec" else self.lm.tok_emb

    def set_input_embedding(self, module):
        if self.kind == "encdec":
            self.lm.tok_emb1 = module
        else:
            self.lm.tok_emb = module

    def _with_virtual(self, ids):
        P = self.n_virtual_tokens
        if not P:
            return ids
        return torch.cat([torch.zeros(ids.shape[0], P, dtype=ids.dtype, device=ids.device), ids], dim=1)

    # ---- teacher-forced forwards (training) ----
    def hidden(self, ids, prefix_len=None):
        """A/B/D: ids (B, T) -> final hidden states (B, T, d). D: bidirectional over the first prefix_len real tokens."""
        P = self.n_virtual_tokens
        if self.kind == "prefix":
            assert prefix_len is not None
            self.lm.cfg.prefix = int(prefix_len) + P
        h = self.lm(self._with_virtual(ids))
        return h[:, P:] if P else h

    def encode(self, src):
        return self.lm.encode(self._with_virtual(src))

    def decode_hidden(self, tgt_in, memory):
        return self.lm.decode(tgt_in, memory)


class GenModel(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def target_logits(self, batch):
        """batch from summ_data.collate -> (logits (N, V) at every labelled position, labels (N,))."""
        bb = self.backbone
        if bb.kind == "encdec":
            h = bb.decode_hidden(batch["dec_in"], bb.encode(batch["src"]))
        else:
            h = bb.hidden(batch["ids"], batch.get("prefix_len"))
        sel = batch["labels"] != -100
        return self.head(h[sel]), batch["labels"][sel]


def build_model(ckpt_path, bcfg, vocab_size=None, load_weights=True, fast_attention=True):
    lm, kind, spec, info = load_phase1(ckpt_path, bcfg.get("max_seq_len_override"), load_weights)
    v = spec["model"]["vocab_size"] if "model" in spec else spec["encoder"]["vocab_size"]
    if vocab_size is not None:
        assert v == vocab_size, f"{ckpt_path}: vocab {v} != tokenizer vocab {vocab_size} (different tokenizer?)"
    if fast_attention:
        (fastattn_ab if kind == "decoder" else fastattn_cd).patch_model(lm)
    head = lm.head                       # tied to tok_emb / tok_emb2 (same Parameter object)
    lm.head = nn.Identity()
    bb = GenBackbone(lm, kind, truncate_layers=bcfg.get("truncate_layers"), dropout=bcfg.get("dropout"))
    info["phase1_max_len"] = (spec["model"]["max_seq_len"] if "model" in spec
                              else {"encoder": spec["encoder"]["max_seq_len"], "decoder": spec["decoder"]["max_seq_len"]})
    return GenModel(bb, head), spec, info


# ---------------------------------------------------------------------------
# generation sessions: prefill(src) once, then step(tok) -> hidden (B, d) of the last position
# ---------------------------------------------------------------------------

class _StepEmbedding(nn.Module):
    """During cached decoding with prompt tuning: run your soft-prompt embedding on [P placeholders + ids] and keep
    only the real positions, so the wrapper's contract (first P positions = prompt) is respected."""

    def __init__(self, wrapped, P):
        super().__init__()
        self.wrapped, self.P = wrapped, P

    def forward(self, ids):
        z = torch.zeros(ids.shape[0], self.P, dtype=ids.dtype, device=ids.device)
        return self.wrapped(torch.cat([z, ids], 1))[:, self.P:]


@contextmanager
def session(model, src, max_new_tokens, dtype, start_id=None):
    """src: (B, S) real source ids (decoder kinds: already include the separator at the end).
    Yields (h_last (B, d), step) where step(tok (B, 1)) -> h_last (B, d)."""
    bb = model.backbone
    lm, P = bb.lm, bb.n_virtual_tokens
    ids = bb._with_virtual(src)
    if bb.kind == "encdec":
        with fastattn_cd.encdec_session(lm, ids, max_new_tokens, dtype, start_id=start_id) as (h0, run):
            yield h0, run
        return
    if bb.kind == "prefix":
        lm.cfg.prefix = ids.shape[1]                     # whole (virtual + source + separator) block bidirectional
        with fastattn_cd.prefix_session(lm, ids, max_new_tokens, dtype) as (h0, run):
            if P:
                emb = bb.get_input_embedding()
                bb.set_input_embedding(_StepEmbedding(emb, P))
                try:
                    yield h0, run
                finally:
                    bb.set_input_embedding(emb)
            else:
                yield h0, run
        return
    with fastattn_ab.kv_cache(lm, ids.shape[0], ids.shape[1] + max_new_tokens, dtype):
        h0 = fastattn_ab.step(lm, ids)[:, -1]
        emb = bb.get_input_embedding()
        if P:
            bb.set_input_embedding(_StepEmbedding(emb, P))
        try:
            yield h0, (lambda tok: fastattn_ab.step(lm, tok)[:, -1])
        finally:
            if P:
                bb.set_input_embedding(emb)

In [ ]:
%%writefile /kaggle/working/phase6/inject.py
"""Generic module-surgery helpers for your fine-tuning techniques (plumbing; declare it).

These only find / replace / freeze modules; they contain no adaptation method. Typical use in your files:

    from inject import freeze, replace_linears, wrap_modules
    freeze(backbone)                                            # base weights frozen
    replace_linears(backbone, lambda name, lin: name.endswith(("q_proj", "v_proj")),
                    lambda name, lin: YourWrapper(lin, ...))    # your LoRA / LoRMA module around nn.Linear
    wrap_modules(backbone, lambda name, m: name.endswith(".ffn"),
                 lambda name, m: YourAdapterWrapper(m, ...))    # your adapter after a sublayer

Linear layer names in the Phase-1 models (relative to backbone, e.g. "lm.blocks.3.attn.q_proj"):
  A/B/D blocks:  attn.q_proj, attn.k_proj, attn.v_proj, attn.out_proj, ffn.net.0 (d->d_ff), ffn.net.2 (d_ff->d)
  C encoder:     lm.encblocks.<i>.attn.{q,k,v,out}_proj, lm.encblocks.<i>.ffn.net.{0,2}
  C decoder:     lm.decblocks.<i>.attn.*, lm.decblocks.<i>.crattn.{q,k,v,out}_proj, lm.decblocks.<i>.ffn.net.{0,2}
Attention projections have no bias (bias=False); FFN linears and LayerNorms do.
Sublayer call signatures: attn(x) (D: attn(x, p)), crattn(x, memory), ffn(x) -> a wrapper should take
(*args, **kwargs) and pass them through to the wrapped module.
"""
import torch.nn as nn


def freeze(module):
    for p in module.parameters():
        p.requires_grad_(False)
    return module


def unfreeze(module):
    for p in module.parameters():
        p.requires_grad_(True)
    return module


def _set_child(root, name, new):
    parent_name, _, child = name.rpartition(".")
    parent = root.get_submodule(parent_name) if parent_name else root
    setattr(parent, child, new)


def list_linears(root):
    return [name for name, m in root.named_modules() if isinstance(m, nn.Linear)]


def replace_linears(root, match, factory):
    """Replace every nn.Linear for which match(name, linear) is True by factory(name, linear). -> replaced names."""
    targets = [(n, m) for n, m in root.named_modules() if isinstance(m, nn.Linear) and match(n, m)]
    for name, lin in targets:
        _set_child(root, name, factory(name, lin))
    return [n for n, _ in targets]


def wrap_modules(root, match, factory):
    """Replace every submodule for which match(name, module) is True by factory(name, module). -> replaced names.
    Matches are collected first, so a wrapper containing the original module is not matched again."""
    targets = [(n, m) for n, m in root.named_modules() if n and match(n, m)]
    for name, mod in targets:
        _set_child(root, name, factory(name, mod))
    return [n for n, _ in targets]


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable, "trainable_pct": round(100 * trainable / max(1, total), 4)}

In [ ]:
%%writefile /kaggle/working/phase6/techniques/__init__.py
"""Technique registry (plumbing). The config's technique.name selects techniques/<name>.py.

Contract for every technique file (YOUR code, course rule 9):
    apply(backbone, **cfg) -> None
        Called once, after the Phase-1 weights are loaded and before the optimizer is built. On entry EVERY backbone
        parameter has requires_grad=True. On exit, requires_grad must be True exactly on what should train
        (plus any modules you inserted). The head is separate and always trained.
        backbone: backbones.Backbone (.lm = your Phase-1 module, .blocks, .kind, .d_model). See inject.py for
        freeze / replace_linears / wrap_modules helpers and the linear-layer names.
    optional: param_groups(model, base_lr) -> list of optimizer param-group dicts (e.g. a different lr for your
        inserted modules). If absent, every trainable backbone parameter uses train.lr and the head uses
        train.head_lr (or train.lr).
"""
import importlib


def get(name):
    return importlib.import_module(f"techniques.{name}")

In [ ]:
%%writefile /kaggle/working/phase6/techniques/full_ft.py
"""Full fine-tuning: every backbone parameter trains (plumbing-level: nothing to implement)."""


def apply(backbone, **cfg):
    assert not cfg, f"full_ft takes no options, got {cfg}"
    for p in backbone.parameters():
        p.requires_grad_(True)

In [ ]:
%%writefile /kaggle/working/phase6/summ_infer.py
#!/usr/bin/env python3
"""Phase-6 summary generation (plumbing; declare it).

Modes
  single  one summary per document from its head+tail-selected source (the run's own format / selection).
  stage1  hierarchical stage 1: the document is packed into sentence chunks of <= src_len tokens and the model writes a
          summary for EVERY chunk (each chunk is a whole "document" to the model) -> {id: [chunk summary, ...]}.
          These files are the source of a stage-2 run (config source = {"type": "stage1", ...}).

Length control (decode["length"]):
  {"mode": "ratio", "ratio": r, "basis": "document" | "input", "min_words": a, "max_words": b, "allow_eos": bool}
      word budget = clip(round(r * words), a, b); words = words of the original document ("document") or of the text
      actually fed to the model ("input": the selected source / the chunk). Generation stops at the budget (and at
      </s> if allow_eos, but never before min_words).
  {"mode": "eos", "min_words": a, "max_words": b}
      stop at </s> (forbidden before min_words) or at max_words.
  Every row is also capped by the context: A/B/D max_len - virtual - src_len - 1 tokens, C the decoder length.
Other decode options: temperature (0 = greedy), top_k, top_p, repetition_penalty, no_repeat_ngram (token n-grams of
the generated text), batch_size, max_new_tokens (optional extra cap).

CLI (uses a finished run's best checkpoint):
  python summ_infer.py --run RUN_DIR --split val|test|train --mode single|stage1 --decode decode.json --out out.json
        [--limit N] [--shard i/n]     (--shard: process every n-th document starting at i; merge with --merge)
  python summ_infer.py --merge out.json part0.json part1.json ...
"""
import argparse
import glob
import json
import math
import os
import sys
import time

import torch

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

import gen_model                      # noqa: E402
import summ_data                      # noqa: E402
import techniques                     # noqa: E402
from a1_tokenizer import ByteBPE      # noqa: E402
from fastattn_ab import ngram_ban, sample_next   # noqa: E402


# ---------------------------------------------------------------------------
# model build / save / load (shared with train_summ.py)
# ---------------------------------------------------------------------------

def build_run_model(cfg, vocab_size, load_weights=True):
    model, spec, info = gen_model.build_model(cfg["phase1_ckpt"], cfg.get("backbone") or {}, vocab_size,
                                              load_weights=load_weights, fast_attention=cfg.get("fast_attention", True))
    for p in model.parameters():
        p.requires_grad_(True)
    tech = cfg["technique"]
    tmod = techniques.get(tech["name"])
    tmod.apply(model.backbone, **{k: v for k, v in tech.items() if k != "name"})
    return model, tmod, info


def save_trainable(model, path):
    """Only what training changed: trainable parameters + every buffer (technique buffers such as QLoRA codes /
    AdaLoRA masks). Frozen parameters are rebuilt from the Phase-1 checkpoint on load."""
    trainable = {n for n, p in model.named_parameters(remove_duplicate=False) if p.requires_grad}
    buffers = {n for n, _ in model.named_buffers()}
    sd = {k: v.detach().cpu() for k, v in model.state_dict().items() if k in trainable or k in buffers}
    torch.save(sd, path + ".tmp")
    os.replace(path + ".tmp", path)


def load_trainable(model, path):
    sd = torch.load(path, map_location="cpu")
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not unexpected, f"{path}: unexpected keys {unexpected[:5]}"
    frozen = {n for n, p in model.named_parameters(remove_duplicate=False) if not p.requires_grad}
    bad = [k for k in missing if k not in frozen]
    assert not bad, f"{path}: missing trainable keys {bad[:5]}"


def resolve_ckpt(path):
    if os.path.exists(path):
        return path
    tail = os.path.join(*path.split(os.sep)[-2:])
    hits = sorted(glob.glob(os.path.join("/kaggle/input", "**", tail), recursive=True))
    assert hits, f"Phase-1 checkpoint {path} not found (looked for */{tail} under /kaggle/input)"
    return hits[0]


def load_run(run_dir, device, vocab_size, ckpt_name="best_model.pt"):
    cfg = json.load(open(os.path.join(run_dir, "config.json")))
    cfg["phase1_ckpt"] = resolve_ckpt(cfg["phase1_ckpt"])
    model, _, _ = build_run_model(cfg, vocab_size, load_weights=True)
    load_trainable(model, os.path.join(run_dir, ckpt_name))
    return model.to(device).eval(), cfg


def special_ids(tok, fmt, kind):
    sp = tok.special_ids
    sep = sp[fmt["sep_token"]] if kind != "encdec" else None
    start = None
    if kind == "encdec":
        st = fmt["start_token"]
        start = None if st == "last_src" else sp[st]
    return sep, start, tok.eot_id


# ---------------------------------------------------------------------------
# batched decoding
# ---------------------------------------------------------------------------

def context_cap(model, src_len):
    bb = model.backbone
    if bb.kind == "encdec":
        return bb.max_tgt_len
    return bb.max_len - bb.n_virtual_tokens - src_len - 1


def word_budget(L, words):
    if L["mode"] == "eos":
        return L.get("max_words")
    b = round(L["ratio"] * words)
    if L.get("min_words"):
        b = max(b, L["min_words"])
    if L.get("max_words"):
        b = min(b, L["max_words"])
    return max(1, b)


@torch.inference_mode()
def generate(model, tok, srcs, budgets, dcfg, sep_id, start_id, eos_id, device, log=None):
    """srcs: list of src_len-id lists; budgets: list of word budgets (None = unlimited). -> list[str] (input order)."""
    model.eval()
    bb = model.backbone
    L = dcfg["length"]
    S = len(srcs[0])
    cap = context_cap(model, S)
    if dcfg.get("max_new_tokens"):
        cap = min(cap, dcfg["max_new_tokens"])
    assert cap > 0, f"no room to generate: src_len {S} fills the context"
    allow_eos = L["mode"] == "eos" or L.get("allow_eos", False)
    min_words = L.get("min_words") or 0
    amp = device.type == "cuda"
    dtype = torch.float16 if amp else torch.float32
    eow = torch.tensor(tok.token_eow, device=device, dtype=torch.bool)
    has_bytes = torch.tensor([len(b) > 0 for b in tok.token_bytes], device=device, dtype=torch.bool)
    n_dec = len(tok.token_bytes)
    banned = torch.tensor([i for i in range(tok.n_special) if not (allow_eos and i == eos_id)], device=device)
    order = sorted(range(len(srcs)), key=lambda i: -(budgets[i] or 10 ** 9))
    out_text = [None] * len(srcs)
    bs = dcfg.get("batch_size") or 16
    t0 = time.time()
    for bi, s0 in enumerate(range(0, len(order), bs)):
        idx = order[s0:s0 + bs]
        B = len(idx)
        src = torch.tensor([srcs[i] + ([sep_id] if bb.kind != "encdec" else []) for i in idx], device=device)
        tgt_words = torch.tensor([budgets[i] if budgets[i] else 10 ** 9 for i in idx], device=device)
        max_new = cap
        words = torch.zeros(B, dtype=torch.long, device=device)
        in_word = torch.zeros(B, dtype=torch.bool, device=device)
        done = torch.zeros(B, dtype=torch.bool, device=device)
        gen = []
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp), \
                gen_model.session(model, src, max_new, dtype, start_id=start_id) as (h, step):
            for t in range(max_new):
                logits = model.head(h).float()
                logits[~in_word, tok.eow_id] = -float("inf")       # bare </w> after a finished word never occurs
                logits[:, n_dec:] = -float("inf")
                if allow_eos and min_words:
                    logits[words < min_words, eos_id] = -float("inf")
                n = dcfg.get("no_repeat_ngram") or 0
                if n > 1 and gen:
                    ban = ngram_ban(torch.stack(gen, 1), n, logits.size(1))
                    ban &= ~(ban | torch.isinf(logits)).all(-1, keepdim=True)
                    logits = logits.masked_fill(ban, -float("inf"))
                rp = dcfg.get("repetition_penalty") or 1.0
                prev = torch.stack(gen, 1) if (rp != 1.0 and gen) else None
                nxt = sample_next(logits, dcfg.get("temperature") or 0, dcfg.get("top_k") or 0, dcfg.get("top_p") or 1.0,
                                  banned, prev, rp)
                nxt = torch.where(done, torch.full_like(nxt, eos_id), nxt)
                gen.append(nxt)
                is_eos = nxt == eos_id
                open_ = in_word | has_bytes[nxt]
                words += (eow[nxt] & open_ & ~done & ~is_eos).long()
                in_word = open_ & ~eow[nxt] & ~is_eos
                done |= is_eos | (words >= tgt_words)
                if bool(done.all()) or t == max_new - 1:
                    break
                h = step(nxt[:, None])
        rows = torch.stack(gen, 1).tolist()
        for r, i in enumerate(idx):
            ids = rows[r]
            if eos_id in ids:
                ids = ids[:ids.index(eos_id)]
            text = " ".join(tok.decode(ids).split())
            if budgets[i]:
                text = " ".join(text.split()[:budgets[i]])
            out_text[i] = text
        if log and (bi + 1) % 10 == 0:
            log(f"generated {min(s0 + bs, len(order))}/{len(order)}", seconds=round(time.time() - t0, 1))
    return out_text


# ---------------------------------------------------------------------------
# document-level entry points
# ---------------------------------------------------------------------------

def words_of(tok, ids):
    return len(tok.decode([t for t in ids if t >= tok.n_special]).split())


def summarize_single(model, tok, cfg, docs, src_enc, dcfg, device, log=None):
    """docs: [{"id", "document"}], src_enc: {id: sentence ids of the run's source text} -> {id: summary}."""
    fmt, sel = cfg["format"], cfg["selection"]
    kind = model.backbone.kind
    sep, start, eos = special_ids(tok, fmt, kind)
    exs, _ = summ_data.build_examples(docs, src_enc, None, fmt, sel, eos)
    L = dcfg["length"]
    budgets = []
    for d, e in zip(docs, exs):
        w = len(d["document"].split()) if L.get("basis", "document") == "document" else words_of(tok, e["src"])
        budgets.append(word_budget(L, w))
    texts = generate(model, tok, [e["src"] for e in exs], budgets, dcfg, sep, start, eos, device, log)
    return {d["id"]: t for d, t in zip(docs, texts)}


def summarize_stage1(model, tok, cfg, docs, doc_enc, dcfg, device, log=None):
    """-> {id: [summary of chunk 0, chunk 1, ...]} (chunks = sentence packing of the document into src_len tokens)."""
    fmt = cfg["format"]
    S = fmt["src_len"]
    kind = model.backbone.kind
    sep, start, eos = special_ids(tok, fmt, kind)
    L = dcfg["length"]
    srcs, budgets, owner = [], [], []
    for d in docs:
        chunks = summ_data.pack_chunks(doc_enc[d["id"]], S)
        dw, total = len(d["document"].split()), sum(map(len, chunks))
        for c in chunks:
            cw = words_of(tok, c)
            # basis "document": the chunk's share of the document's word budget (ratio x chunk words, same thing)
            budgets.append(word_budget(L, cw if L.get("basis", "document") == "input" else dw * len(c) / max(1, total)))
            srcs.append(summ_data.pad_left(c, S))
            owner.append(d["id"])
    if log:
        log("stage1 chunks", docs=len(docs), chunks=len(srcs), per_doc=round(len(srcs) / max(1, len(docs)), 2))
    texts = generate(model, tok, srcs, budgets, dcfg, sep, start, eos, device, log)
    out = {d["id"]: [] for d in docs}
    for o, t in zip(owner, texts):
        out[o].append(t)
    return out


def write_prediction(summaries, test_docs, path):
    pred = [{"id": d["id"], "summary": summaries[d["id"]]} for d in test_docs]
    assert len(pred) == len(test_docs) and len({p["id"] for p in pred}) == len(pred)
    assert all(isinstance(p["summary"], str) for p in pred)
    json.dump(pred, open(path, "w"), ensure_ascii=False)
    return pred


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def source_docs(cfg, split_docs, split):
    """Documents whose 'document' field is the run's SOURCE text (stage-2 runs: joined stage-1 chunk summaries)."""
    src = cfg.get("source") or {"type": "document"}
    if src["type"] == "document":
        return split_docs
    s1 = json.load(open(src[split]))
    missing = [d["id"] for d in split_docs if d["id"] not in s1]
    assert not missing, f"stage-1 file {src[split]} lacks {len(missing)} {split} documents (e.g. {missing[:3]})"
    return [{**d, "document": " ".join(s1[d["id"]])} for d in split_docs]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run")
    ap.add_argument("--split", choices=["train", "val", "test"])
    ap.add_argument("--mode", choices=["single", "stage1"], default="single")
    ap.add_argument("--decode", help="JSON file or inline JSON; default = the run config's decode block")
    ap.add_argument("--out")
    ap.add_argument("--limit", type=int)
    ap.add_argument("--shard", default="0/1")
    ap.add_argument("--merge", nargs="+")
    ap.add_argument("--data-dir")
    ap.add_argument("--zip", action="store_true", help="test split, single mode: also write submission zip next to --out")
    a = ap.parse_args()
    if a.merge:
        out, parts = a.merge[0], a.merge[1:]
        merged = {}
        for p in parts:
            merged.update(json.load(open(p)))
        json.dump(merged, open(out, "w"), ensure_ascii=False)
        print(f"merged {len(parts)} parts -> {out} ({len(merged)} documents)")
        return
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    cfg = json.load(open(os.path.join(a.run, "config.json")))
    tok = ByteBPE.from_json(resolve_ckpt(cfg["tokenizer"]) if not os.path.exists(cfg["tokenizer"]) else cfg["tokenizer"])
    model, cfg = load_run(a.run, device, tok.get_vocab_size())
    dcfg = cfg["decode"]
    if a.decode:
        dcfg = json.loads(a.decode) if a.decode.strip().startswith("{") else json.load(open(a.decode))
    data_dir = a.data_dir or cfg["data_dir"]
    train, val, test, _ = summ_data.load_splits(data_dir, cfg["split"]["val_docs"], cfg["split"]["seed"])
    docs = {"train": train, "val": val, "test": test}[a.split]
    if a.limit:
        docs = docs[:a.limit]
    i, n = map(int, a.shard.split("/"))
    docs = docs[i::n]
    log = lambda msg, **kv: print(json.dumps({"msg": msg, **kv}), flush=True)
    t0 = time.time()
    if a.mode == "stage1":
        enc = summ_data.encode_sentences(tok, docs, "document")
        res = summarize_stage1(model, tok, cfg, docs, enc, dcfg, device, log)
    else:
        sdocs = source_docs(cfg, docs, a.split)
        enc = summ_data.encode_sentences(tok, sdocs, "document")
        res = summarize_single(model, tok, cfg, docs, enc, dcfg, device, log)
    json.dump(res, open(a.out, "w"), ensure_ascii=False)
    log("done", docs=len(docs), out=a.out, seconds=round(time.time() - t0, 1))
    if a.zip and a.split == "test" and a.mode == "single" and n == 1:
        import zipfile
        pred_path = os.path.join(os.path.dirname(os.path.abspath(a.out)), "prediction.json")
        write_prediction(res, docs, pred_path)
        zp = os.path.splitext(a.out)[0] + ".zip"
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(pred_path, arcname="prediction.json")
        log("zip", path=zp)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase6/score_summ.py
#!/usr/bin/env python3
"""Local scorer for Phase-6 summaries (plumbing; declare it). Facts only: metrics + length / repetition statistics.

  python score_summ.py PRED.json [PRED2.json ...] --refs val_refs.json [--bert-model roberta-large] [--device cuda]

PRED.json: {id: summary} (summ_infer / train_summ output) or the list format [{"id", "summary"}].
--refs: {id: reference summary} (train_summ writes <run>/val_refs.json).
Metrics
  BERTScore F1 (primary)  bert_score, default roberta-large, no baseline rescaling. The official "legal BERTScore"
                          model is unknown: pass --bert-model to match it once known. Inputs longer than the model's
                          512-token limit are truncated by bert_score (most references are longer: compare runs only
                          with the same settings). --no-bert skips it.
  ROUGE-1 / ROUGE-L F1    rouge_score, with the Porter stemmer (--no-stem to disable).
  length                  mean words of predictions vs references, mean pred/ref word ratio.
  repetition              rep4: mean share of repeated word 4-grams per prediction; loop%: predictions where some
                          word 4-gram occurs 3+ times.
"""
import argparse
import json
import statistics
from collections import Counter


def load_pred(path):
    d = json.load(open(path))
    if isinstance(d, list):
        d = {x["id"]: x["summary"] for x in d}
    return {k: (v if isinstance(v, str) else " ".join(v)) for k, v in d.items()}


def rep_stats(text):
    w = text.split()
    g = Counter(tuple(w[i:i + 4]) for i in range(len(w) - 3))
    n = sum(g.values())
    return (sum(c - 1 for c in g.values()) / n if n else 0.0), any(c >= 3 for c in g.values())


def score(preds, refs, bert_model=None, device=None, stem=True, batch_size=16):
    ids = sorted(refs)
    missing = [i for i in ids if i not in preds]
    assert not missing, f"{len(missing)} reference ids have no prediction (e.g. {missing[:3]})"
    P = [preds[i] for i in ids]
    R = [refs[i] for i in ids]
    from rouge_score import rouge_scorer
    sc = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=stem)
    r1, rl = [], []
    for p, r in zip(P, R):
        s = sc.score(r, p)
        r1.append(s["rouge1"].fmeasure); rl.append(s["rougeL"].fmeasure)
    out = {"n": len(ids), "rouge1_f1": round(statistics.mean(r1), 4), "rougeL_f1": round(statistics.mean(rl), 4)}
    if bert_model:
        from bert_score import score as bscore
        _, _, F = bscore(P, R, model_type=bert_model, device=device, batch_size=batch_size, verbose=False)
        out["bertscore_f1"] = round(float(F.mean()), 4)
        out["bert_model"] = bert_model
    pw = [len(p.split()) for p in P]
    rw = [len(r.split()) for r in R]
    reps = [rep_stats(p) for p in P]
    out.update(pred_words=round(statistics.mean(pw), 1), ref_words=round(statistics.mean(rw), 1),
               len_ratio=round(statistics.mean(a / max(1, b) for a, b in zip(pw, rw)), 3),
               empty=sum(not p.strip() for p in P),
               rep4=round(statistics.mean(r for r, _ in reps), 4),
               loop_pct=round(100 * sum(l for _, l in reps) / len(reps), 1))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("preds", nargs="+")
    ap.add_argument("--refs", required=True)
    ap.add_argument("--bert-model", default="roberta-large")
    ap.add_argument("--no-bert", action="store_true")
    ap.add_argument("--no-stem", action="store_true")
    ap.add_argument("--device")
    ap.add_argument("--out", help="append one JSON line per file")
    a = ap.parse_args()
    refs = json.load(open(a.refs))
    for p in a.preds:
        preds = load_pred(p)
        res = {"file": p, **score(preds, refs, None if a.no_bert else a.bert_model, a.device, not a.no_stem)}
        print(json.dumps(res), flush=True)
        if a.out:
            with open(a.out, "a") as f:
                f.write(json.dumps(res) + "\n")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/phase6/train_summ.py
#!/usr/bin/env python3
"""Fine-tune one Phase-1 model for abstractive summarization and write its Codabench submission (plumbing; declare it).

  python train_summ.py run_config.json

Pipeline: train.json -> clean (empty summaries, duplicate documents, test overlap removed) -> seeded validation hold-out
(split.val_docs) -> sentence-level ByteBPE (cached) -> source = head+tail selection of src_len tokens (summ_data) ->
target = reference truncated to tgt_len (whole sentences, + </s>) -> Phase-1 model WITH its LM head (gen_model) + YOUR
technique (techniques/<name>.py) -> token cross-entropy on the target tokens only (teacher forcing).

Source text (cfg["source"]):
  {"type": "document"}                                              the judgment itself (single-summary / stage-1 runs)
  {"type": "stage1", "train": f1, "val": f2, "test": f3}            stage 2 of the hierarchical method: the source is the
      concatenation of the stage-1 chunk summaries (files written by summ_infer.py --mode stage1 or by "after.stage1"
      below). Train documents missing from f1 are skipped (a stage-1 subset is allowed); val / test must be complete.

Checkpoint selection: lowest validation loss (teacher-forced, mean per target token), evaluated every
train.eval_every_steps and at the end of every epoch. Early stop: train.patience evaluations without improvement.

After training (best checkpoint): summaries for eval_generate.val_docs validation documents (scored with ROUGE-1/L and,
if eval_generate.bert_model is set, BERTScore) and for all test documents -> prediction.json (list format) + zip.
Optional "after": {"stage1": {"train_docs": N | "all", "decode": {...}}} also writes stage-1 chunk summaries of the
first N train documents, all validation and all test documents (stage1_train/val/test.json) for a stage-2 run.

KD (cfg["kd"] = {"teacher_run": <finished run dir>, ...your kd options}): the teacher (frozen) runs on the same documents
in its own input format; target tokens must be identical (same tgt_len / eos_on_truncated / target_trunc / source,
checked). YOUR techniques/kd.py kd_loss(student_logits (1, N, V), teacher_logits (1, N, V), labels (1, N), **opts) over
every target token replaces the cross-entropy. The student must have fewer parameters than the teacher (checked).
Sequence-level KD alternative: cfg["target_override"] = {id: text} file (e.g. teacher summaries of the train documents
from summ_infer.py) replaces the reference summaries of TRAIN documents only (validation keeps the references).

Outputs in <out_dir>/<name>/: config.json, log.jsonl, best_model.pt (trainable parameters + buffers only), val_refs.json,
val_generations.json, val_scores.json, test_summaries.json, prediction.json, submission_<name>.zip, summary.json.
"""
import inspect
import json
import math
import os
import random
import sys
import time
import zipfile

import numpy as np
import torch
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

import score_summ                      # noqa: E402
import summ_data                       # noqa: E402
import summ_infer                      # noqa: E402
import techniques                      # noqa: E402
from a1_tokenizer import ByteBPE       # noqa: E402
from inject import count_params        # noqa: E402

REQUIRED = {
    "format": ["src_len", "tgt_len", "eos_on_truncated"],
    "selection": ["strategy", "head_frac"],
    "split": ["val_docs"],
    "train": ["epochs", "batch_examples", "micro_examples", "lr", "weight_decay", "warmup_frac", "schedule",
              "grad_clip", "seed"],
}


def check_config(cfg, kind):
    missing = [f"{s}.{k}" for s, ks in REQUIRED.items() for k in ks if (cfg.get(s) or {}).get(k) is None]
    missing += [f"technique.name"] if (cfg.get("technique") or {}).get("name") is None else []
    if kind == "encdec":
        missing += ["format.start_token"] if cfg["format"].get("start_token") is None else []
    elif kind is not None:
        missing += ["format.sep_token"] if cfg["format"].get("sep_token") is None else []
    d = cfg.get("decode") or {}
    L = d.get("length") or {}
    if L.get("mode") is None:
        missing.append("decode.length.mode")
    elif L["mode"] == "ratio":
        missing += [f"decode.length.{k}" for k in ("ratio", "basis") if L.get(k) is None]
    assert not missing, "fill these config values first: " + ", ".join(missing)
    assert cfg["selection"]["strategy"] == "headtail", "selection.strategy: only 'headtail' is implemented"
    assert 0.0 <= cfg["selection"]["head_frac"] <= 1.0
    assert L["mode"] in ("ratio", "eos") and (L["mode"] == "eos" or L["basis"] in ("document", "input"))
    assert cfg["format"].get("target_trunc", "sentence") in ("sentence", "token")
    tr = cfg["train"]
    assert tr["batch_examples"] % tr["micro_examples"] == 0, "train.batch_examples must be a multiple of micro_examples"
    assert tr["schedule"] in ("linear", "cosine", "constant")
    tech = cfg["technique"]
    unset = [f"technique.{k}" for k, v in tech.items() if v is None]
    if cfg.get("kd"):
        unset += [f"kd.{k}" for k, v in cfg["kd"].items() if v is None]
    assert not unset, "fill these config values first: " + ", ".join(unset)
    sig = inspect.signature(techniques.get(tech["name"]).apply)
    try:
        sig.bind(None, **{k: v for k, v in tech.items() if k != "name"})
    except TypeError as e:
        raise AssertionError(f"technique '{tech['name']}' options do not fit apply{sig}: {e}")


class Logger:
    def __init__(self, path):
        self.path = path

    def __call__(self, msg, **kv):
        rec = {"t": round(time.time(), 1), "msg": msg, **kv}
        print(json.dumps(rec), flush=True)
        with open(self.path, "a") as f:
            f.write(json.dumps(rec) + "\n")


def lr_factor(step, total, warmup, schedule):
    if warmup and step < warmup:
        return (step + 1) / warmup
    if schedule == "constant":
        return 1.0
    prog = min(1.0, (step - warmup) / max(1, total - warmup))
    return 1.0 - prog if schedule == "linear" else 0.5 * (1 + math.cos(math.pi * prog))


def micro_batches(exs, micro, shuffle, rng):
    order = sorted(range(len(exs)), key=lambda i: len(exs[i].get("tgt", ())))   # similar target lengths together
    mbs = [order[i:i + micro] for i in range(0, len(order), micro)]
    if shuffle:
        rng.shuffle(mbs)
    return mbs


def check_lengths(model, fmt):
    bb = model.backbone
    S, Tt, P = fmt["src_len"], fmt["tgt_len"], bb.n_virtual_tokens
    if bb.kind == "encdec":
        assert S + P <= bb.max_len, f"src_len {S} + {P} virtual tokens > encoder length {bb.max_len}"
        assert Tt <= bb.max_tgt_len, f"tgt_len {Tt} > decoder length {bb.max_tgt_len} (backbone.max_seq_len_override)"
    else:
        assert P + S + Tt <= bb.max_len, \
            f"{P} virtual + src_len {S} + tgt_len {Tt} = {P + S + Tt} > context {bb.max_len} (separator included)"


def load_texts(cfg, tok, log, smoke):
    D, cache = cfg["data_dir"], cfg.get("cache_dir")
    train, val, test, rep = summ_data.load_splits(D, cfg["split"]["val_docs"], cfg["split"].get("seed", 0))
    src = cfg.get("source") or {"type": "document"}
    if src["type"] == "stage1":
        s1 = {k: json.load(open(src[k])) for k in ("train", "val", "test")}
        n0 = len(train)
        train = [d for d in train if d["id"] in s1["train"]]
        rep["train_with_stage1"] = len(train)
        rep["train_skipped_no_stage1"] = n0 - len(train)
        for k, docs in (("val", val), ("test", test)):
            miss = [d["id"] for d in docs if d["id"] not in s1[k]]
            assert not miss or smoke, f"stage-1 {k} file lacks {len(miss)} documents"
        srcdocs = {k: [{"id": d["id"], "document": " ".join(s1[k].get(d["id"], []))} for d in docs]
                   for k, docs in (("train", train), ("val", val), ("test", test))}
    else:
        srcdocs = {"train": train, "val": val, "test": test}
    if cfg.get("target_override"):
        ov = json.load(open(cfg["target_override"]))
        n0 = len(train)
        train = [{**d, "summary": ov[d["id"]]} for d in train if d["id"] in ov and ov[d["id"]].strip()]
        srcdocs["train"] = [d for d in srcdocs["train"] if d["id"] in {x["id"] for x in train}]
        rep["target_override"] = cfg["target_override"]
        rep["train_with_override"] = len(train)
        rep["train_skipped_no_override"] = n0 - len(train)
    if smoke:
        n = smoke.get("docs", 16)
        train, val, test = train[:n], val[:4], test[:4]
        keep = {d["id"] for d in train + val + test}
        srcdocs = {k: [d for d in v if d["id"] in keep] for k, v in srcdocs.items()}
    tag = src["type"] if src["type"] == "document" else \
        "stage1_" + summ_data.hashlib.md5(json.dumps(src, sort_keys=True).encode()).hexdigest()[:8]
    cp = lambda name: os.path.join(cache, f"{name}.pkl") if cache and not smoke else None
    enc = {k: summ_data.encode_sentences(tok, srcdocs[k], "document", cp(f"{tag}_{k}_src")) for k in srcdocs}
    tgt_tag = "override" if cfg.get("target_override") else "ref"
    tgt = {"train": summ_data.encode_sentences(tok, train, "summary", cp(f"{tgt_tag}_train_tgt")),
           "val": summ_data.encode_sentences(tok, val, "summary", cp("ref_val_tgt"))}
    return {"train": train, "val": val, "test": test}, srcdocs, enc, tgt, rep


@torch.no_grad()
def val_loss(model, exs, micro, kind, sep, start, device, amp):
    model.eval()
    tot, n = 0.0, 0
    for idx in micro_batches(exs, micro, False, None):
        b = summ_data.collate([exs[i] for i in idx], kind, sep, start, device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            lg, lab = model.target_logits(b)
        tot += F.cross_entropy(lg.float(), lab, reduction="sum").item()
        n += lab.numel()
    model.train()
    return tot / max(1, n)


def main(cfg_path):
    cfg = json.load(open(cfg_path))
    name, tr, fmt = cfg["name"], cfg["train"], cfg["format"]
    out = os.path.join(cfg["out_dir"], name)
    os.makedirs(out, exist_ok=True)
    log = Logger(os.path.join(out, "log.jsonl"))
    seed = tr.get("seed") or 0
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    t_start = time.time()
    deadline = t_start + 60 * tr["max_minutes"] if tr.get("max_minutes") else None
    smoke = cfg.get("smoke")          # {"steps": n, "docs": m}: real model, few documents and steps; not submittable

    check_config(cfg, None)
    tok = ByteBPE.from_json(cfg["tokenizer"])
    cfg["phase1_ckpt"] = summ_infer.resolve_ckpt(cfg["phase1_ckpt"])
    model, tmod, info = summ_infer.build_run_model(cfg, tok.get_vocab_size())
    kind = model.backbone.kind
    check_config(cfg, kind)
    json.dump(cfg, open(os.path.join(out, "config.json"), "w"), indent=2)
    if cfg.get("init_from_run"):
        init_run = cfg["init_from_run"]
        assert os.path.abspath(init_run) != os.path.abspath(out)
        icfg = json.load(open(os.path.join(init_run, "config.json")))
        for k in ("technique", "backbone"):
            assert (icfg.get(k) or {}) == (cfg.get(k) or {}), f"init_from_run: '{k}' differs from {init_run}"
        summ_infer.load_trainable(model, os.path.join(init_run, "best_model.pt"))
        info["init_from_run"] = init_run
    model.to(device)
    check_lengths(model, fmt)
    sep, start, eos = summ_infer.special_ids(tok, fmt, kind)

    docs, srcdocs, enc, tgt, drep = load_texts(cfg, tok, log, smoke)
    exs, est = {}, {}
    for s in ("train", "val"):
        exs[s], est[s] = summ_data.build_examples(srcdocs[s], enc[s], tgt[s], fmt, cfg["selection"], eos)
    exs["test"], est["test"] = summ_data.build_examples(srcdocs["test"], enc["test"], None, fmt, cfg["selection"], eos)
    log("data", **drep, **{f"{s}_{k}": v for s in est for k, v in est[s].items()}, kind=kind,
        src_len=fmt["src_len"], tgt_len=fmt["tgt_len"])
    params = count_params(model)
    log("model", **info, **params, max_len=model.backbone.max_len,
        max_tgt_len=model.backbone.max_tgt_len, virtual_tokens=model.backbone.n_virtual_tokens)

    teacher = None
    if cfg.get("kd"):
        kd = {k: v for k, v in cfg["kd"].items() if k != "teacher_run"}
        tdir = cfg["kd"]["teacher_run"]
        teacher, tcfg = summ_infer.load_run(tdir, device, tok.get_vocab_size())
        for p in teacher.parameters():
            p.requires_grad_(False)
        t_total = count_params(teacher)["total"]
        assert params["total"] < t_total, f"KD student ({params['total']:,}) must be smaller than the teacher ({t_total:,})"
        for k in ("tgt_len", "eos_on_truncated", "target_trunc"):
            assert tcfg["format"].get(k) == fmt.get(k), f"KD: format.{k} differs from the teacher ({tcfg['format'].get(k)})"
        assert (tcfg.get("source") or {"type": "document"}) == (cfg.get("source") or {"type": "document"}), "KD: source differs"
        t_kind = teacher.backbone.kind
        check_lengths(teacher, tcfg["format"])
        t_sep, t_start_id, _ = summ_infer.special_ids(tok, tcfg["format"], t_kind)
        t_exs, _ = summ_data.build_examples(srcdocs["train"], enc["train"], tgt["train"], tcfg["format"],
                                            tcfg["selection"], eos)
        t_by_id = {e["id"]: e for e in t_exs}
        kd_loss = techniques.get("kd").kd_loss
        log("teacher", run=tcfg["name"], kind=t_kind, params=t_total, student_params=params["total"])

    if hasattr(tmod, "param_groups"):
        groups = tmod.param_groups(model, tr["lr"])
    else:
        seen, ps = set(), []
        for p in model.parameters():
            if p.requires_grad and id(p) not in seen:
                seen.add(id(p)); ps.append(p)
        groups = [{"params": ps, "lr": tr["lr"]}]
    for g in groups:
        g["params"] = list(g["params"]); g["base_lr"] = g["lr"]
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=tr["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    extra_loss, on_step = getattr(tmod, "extra_loss", None), getattr(tmod, "on_step", None)

    rng = random.Random(seed)
    micro, accum = tr["micro_examples"], tr["batch_examples"] // tr["micro_examples"]
    steps_per_epoch = math.ceil(len(micro_batches(exs["train"], micro, False, None)) / accum)
    total = steps_per_epoch * tr["epochs"]
    warmup = int(tr["warmup_frac"] * total)
    im = tr.get("infer_micro_examples") or micro
    log("schedule", steps_per_epoch=steps_per_epoch, total_steps=total, warmup_steps=warmup,
        train_target_tokens=int(sum(len(e["tgt"]) for e in exs["train"])))
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    best, best_at, bad_evals, step, seen_tok, history = float("inf"), None, 0, 0, 0, []
    stop = False
    t_train = time.time()

    def evaluate(tag, rec):
        nonlocal best, best_at, bad_evals
        t_ev = time.time()
        vl = val_loss(model, exs["val"], im, kind, sep, start, device, amp)
        rec.update(at=tag, val_loss=round(vl, 4), eval_s=round(time.time() - t_ev, 1))
        if vl < best:
            best, best_at, bad_evals = vl, tag, 0
            summ_infer.save_trainable(model, os.path.join(out, "best_model.pt"))
            rec["saved_best"] = True
        else:
            bad_evals += 1
        history.append(rec)
        log("eval", **rec)

    eval_every = tr.get("eval_every_steps")
    for epoch in range(1, tr["epochs"] + 1):
        model.train()
        mbs = micro_batches(exs["train"], micro, True, rng)
        run_loss, n_tok = 0.0, 0
        for i in range(0, len(mbs), accum):
            group = mbs[i:i + accum]
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_factor(step, total, warmup, tr["schedule"])
            n_step_tok = sum(len(exs["train"][j]["tgt"]) for idx in group for j in idx)
            for idx in group:
                batch_ex = [exs["train"][j] for j in idx]
                b = summ_data.collate(batch_ex, kind, sep, start, device)
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    lg, lab = model.target_logits(b)
                lg = lg.float()
                if teacher is not None:
                    tb = summ_data.collate([t_by_id[e["id"]] for e in batch_ex], t_kind, t_sep, t_start_id, device)
                    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                        tlg, tlab = teacher.target_logits(tb)
                    assert torch.equal(tlab, lab), "KD: teacher / student target tokens differ"
                    loss = kd_loss(lg[None], tlg.float()[None], lab[None], **kd)
                else:
                    loss = F.cross_entropy(lg, lab)
                if extra_loss is not None:
                    loss = loss + extra_loss(model)
                scaler.scale(loss * lab.numel() / n_step_tok).backward()     # mean over the step's target tokens
                run_loss += loss.item() * lab.numel(); n_tok += lab.numel()
                seen_tok += b["ids"].numel() if "ids" in b else b["src"].numel() + b["dec_in"].numel()
            if tr["grad_clip"]:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], tr["grad_clip"])
            scaler.step(opt); scaler.update()
            step += 1
            if on_step is not None:          # before zero_grad: your hook can still read p.grad (AdaLoRA importance)
                on_step(model, step, total)
            opt.zero_grad(set_to_none=True)
            if smoke and step >= smoke["steps"]:
                stop = True; break
            if step % (tr.get("log_every") or 50) == 0:
                el = time.time() - t_train
                log("train", epoch=epoch, step=step, of=total, loss=round(run_loss / max(1, n_tok), 4),
                    lr=opt.param_groups[0]["lr"], tok_s=round(seen_tok / el),
                    eta_min=round((total - step) * el / step / 60, 1))
            if eval_every and step % eval_every == 0 and step % steps_per_epoch != 0:
                evaluate(f"e{epoch}_step{step}", dict(epoch=epoch, step=step, train_loss=round(run_loss / max(1, n_tok), 4)))
                if tr.get("patience") and bad_evals >= tr["patience"]:
                    log("early stop", bad_evals=bad_evals); stop = True; break
            if deadline and time.time() > deadline:
                log("max_minutes reached: stopping inside the epoch", step=step); stop = True; break
        evaluate(f"e{epoch}_end", dict(epoch=epoch, step=step, train_loss=round(run_loss / max(1, n_tok), 4)))
        if stop or (tr.get("patience") and bad_evals >= tr["patience"]):
            break
    train_s = time.time() - t_train
    peak = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else None
    del opt
    if teacher is not None:
        del teacher
        torch.cuda.empty_cache()

    # ---- generation with the best checkpoint ----
    summ_infer.load_trainable(model, os.path.join(out, "best_model.pt"))
    model.eval()
    dcfg = cfg["decode"]
    eg = cfg.get("eval_generate") or {}
    n_val = len(docs["val"]) if eg.get("val_docs") in (None, "all") else int(eg["val_docs"])
    vdocs = docs["val"][:n_val]
    refs = {d["id"]: d["summary"] for d in vdocs}
    json.dump(refs, open(os.path.join(out, "val_refs.json"), "w"), ensure_ascii=False)
    vsrc = {d["id"]: enc["val"][d["id"]] for d in vdocs}
    t0 = time.time()
    vgen = summ_infer.summarize_single(model, tok, cfg, vdocs, vsrc, dcfg, device, log) if vdocs else {}
    val_gen_s = time.time() - t0
    json.dump(vgen, open(os.path.join(out, "val_generations.json"), "w"), ensure_ascii=False)
    vsc = {}
    if vdocs:
        vsc = score_summ.score(vgen, refs, eg.get("bert_model"), "cuda" if amp else "cpu")
        json.dump(vsc, open(os.path.join(out, "val_scores.json"), "w"), indent=2)
        log("val generation", **vsc, seconds=round(val_gen_s, 1))
    t0 = time.time()
    tgen = summ_infer.summarize_single(model, tok, cfg, docs["test"], enc["test"], dcfg, device, log)
    test_gen_s = time.time() - t0
    json.dump(tgen, open(os.path.join(out, "test_summaries.json"), "w"), ensure_ascii=False)
    pred_path = os.path.join(out, "prediction.json")
    summ_infer.write_prediction(tgen, docs["test"], pred_path)
    zip_path = os.path.join(out, f"submission_{name}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")

    after = (cfg.get("after") or {}).get("stage1")
    s1_info = None
    if after:
        assert (cfg.get("source") or {"type": "document"})["type"] == "document", "stage-1 outputs need a document-source run"
        n_tr = len(docs["train"]) if after.get("train_docs") in (None, "all") else int(after["train_docs"])
        s1_info = {}
        for s, dd in (("train", docs["train"][:n_tr]), ("val", docs["val"]), ("test", docs["test"])):
            denc = enc[s] if s != "train" else {d["id"]: enc["train"][d["id"]] for d in dd}
            t0 = time.time()
            res = summ_infer.summarize_stage1(model, tok, cfg, dd, denc, after["decode"], device, log)
            p = os.path.join(out, f"stage1_{s}.json")
            json.dump(res, open(p, "w"), ensure_ascii=False)
            s1_info[s] = {"docs": len(res), "chunks": sum(map(len, res.values())), "seconds": round(time.time() - t0, 1)}
            log("stage1 written", split=s, path=p, **s1_info[s])

    summary = dict(name=name, phase1=info, kind=kind, technique=cfg["technique"], kd=bool(cfg.get("kd")),
                   format=fmt, selection=cfg["selection"], source=cfg.get("source") or {"type": "document"},
                   decode=dcfg, params=params, best_val_loss=round(best, 4), best_at=best_at, history=history,
                   val_scores=vsc, val_generated=len(vgen), train_seconds=round(train_s, 1),
                   train_tok_s=round(seen_tok / max(train_s, 1e-9)), peak_mem_GB=round(peak, 2) if peak else None,
                   val_gen_seconds=round(val_gen_s, 1), test_gen_seconds=round(test_gen_s, 1),
                   test_gen_s_per_doc=round(test_gen_s / max(1, len(docs["test"])), 3), stage1=s1_info,
                   data=drep, smoke=bool(smoke), total_seconds=round(time.time() - t_start, 1))
    json.dump(summary, open(os.path.join(out, "summary.json"), "w"), indent=2)
    log("done", name=name, best_val_loss=round(best, 4), best_at=best_at, **{f"val_{k}": v for k, v in vsc.items()
        if k in ("rouge1_f1", "rougeL_f1", "bertscore_f1", "pred_words")}, zip=zip_path,
        train_min=round(train_s / 60, 1), trainable=params["trainable"])


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/phase6/run_many.py
#!/usr/bin/env python3
"""Run several train_summ.py configs over the available GPUs IN PARALLEL (plumbing; declare it).

  python run_many.py --gpus 0,1 configs/B_full.json configs/C_full.json ...

One worker per GPU; every worker runs one training process at a time on its own GPU (CUDA_VISIBLE_DEVICES), so with
--gpus 0,1 two fine-tuning runs train simultaneously, one per T4. Scheduling uses a SHARED queue: whenever a GPU becomes
free it takes the first runnable config in list order (no GPU sits idle while work remains). A config with "kd" (or "wait_for_run": a stage-1 run) is only
runnable once that run has written summary.json; until then the free GPU takes the next job instead. If the teacher
fails (or is not part of this batch and has no summary.json), the KD run is marked FAILED instead of waiting forever.
With --stop-at UNIX_TIME a run is only STARTED if it can finish before then (its train.max_minutes + --margin minutes,
or --default-minutes when max_minutes is unset); runs that no longer fit are marked SKIPPED, so the session limit never
kills a run halfway.

Logs: <out_dir>/<name>.stdout (+ <out_dir>/<name>/log.jsonl). A status line every --every seconds, a summary at the end.
Exit code 1 if any run failed.
"""
import argparse
import json
import os
import subprocess
import sys
import threading
import time

HERE = os.path.dirname(os.path.abspath(__file__))


class Scheduler:
    def __init__(self, cfg_paths, stop_at=None, margin=10, default_minutes=60):
        self.stop_at, self.margin, self.default_minutes = stop_at, margin, default_minutes
        self.lock = threading.Lock()
        self.cfgs = {}
        for p in cfg_paths:
            c = json.load(open(p))
            self.cfgs[c["name"]] = (p, c)
        self.pending = list(self.cfgs)                        # names, in the order given
        self.status = {n: "queued" for n in self.pending}
        self.started = {}

    def _deps(self, cfg):
        """runs this config needs finished first: its KD teacher and/or its stage-1 run (wait_for_run)."""
        dirs = []
        if cfg.get("kd"):
            dirs.append(cfg["kd"]["teacher_run"])
        if cfg.get("wait_for_run"):
            dirs.append(cfg["wait_for_run"])
        return [(os.path.basename(d.rstrip("/")), os.path.join(d, "summary.json")) for d in dirs]

    def next_job(self):
        """-> (name, path, cfg) to run now | "wait" (only KD runs whose teacher is still training) | None (nothing left)."""
        with self.lock:
            if not self.pending:
                return None
            for name in list(self.pending):
                path, cfg = self.cfgs[name]
                blocked, failed = None, None
                for teacher, done in self._deps(cfg):
                    if os.path.exists(done):
                        continue
                    t_status = self.status.get(teacher)
                    if t_status is None or t_status.startswith(("FAILED", "SKIPPED", "done")):
                        failed = teacher
                    else:
                        blocked = teacher
                if failed:
                    self.pending.remove(name)
                    self.status[name] = f"FAILED (dependency {failed} has no summary.json)"
                    continue
                if blocked:
                    self.status[name] = f"waiting for {blocked}"
                    continue
                if self.stop_at is not None:
                    need = ((cfg.get("train", {}).get("max_minutes") or cfg.get("est_minutes") or self.default_minutes) + self.margin) * 60
                    if time.time() + need > self.stop_at:
                        self.pending.remove(name)
                        self.status[name] = f"SKIPPED (needs ~{need / 60:.0f} min, session limit too close)"
                        continue
                self.pending.remove(name)
                return name, path, cfg
            return "wait" if self.pending else None

    def worker(self, gpu):
        while True:
            job = self.next_job()
            if job is None:
                return
            if job == "wait":
                time.sleep(10)
                continue
            name, path, cfg = job
            os.makedirs(cfg["out_dir"], exist_ok=True)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2")
            with self.lock:
                self.status[name] = f"running on GPU {gpu}"
                self.started[name] = time.time()
            with open(os.path.join(cfg["out_dir"], f"{name}.stdout"), "w") as log:
                rc = subprocess.call([sys.executable, os.path.join(HERE, "train_summ.py"), path], env=env,
                                     stdout=log, stderr=subprocess.STDOUT, cwd=os.getcwd())
            mins = (time.time() - self.started[name]) / 60
            if rc != 0:                                  # show WHY it failed right in the main log
                try:
                    tail = open(os.path.join(cfg["out_dir"], f"{name}.stdout"), errors="replace").read().strip().splitlines()[-25:]
                    print(f"\n----- {name} FAILED (exit {rc}): last lines of its output -----\n" + "\n".join(tail) +
                          "\n-----", flush=True)
                except OSError:
                    pass
            with self.lock:
                self.status[name] = f"done on GPU {gpu} in {mins:.1f} min" if rc == 0 else \
                    f"FAILED (exit {rc}) on GPU {gpu} after {mins:.1f} min"


def last_log_line(cfg):
    logf = os.path.join(cfg["out_dir"], cfg["name"], "log.jsonl")
    if not os.path.exists(logf):
        return None
    lines = open(logf).read().strip().splitlines()
    return lines[-1][:250] if lines else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("configs", nargs="+")
    ap.add_argument("--gpus", default="0")
    ap.add_argument("--every", type=int, default=300)
    ap.add_argument("--stop-at", type=float, default=None, help="unix time: do not start runs that cannot finish by then")
    ap.add_argument("--margin", type=float, default=10, help="minutes added to max_minutes for evaluation + prediction")
    ap.add_argument("--default-minutes", type=float, default=60, help="assumed run length when max_minutes is unset")
    args = ap.parse_args()
    gpus = [g for g in args.gpus.split(",") if g != ""]
    sched = Scheduler(args.configs, args.stop_at, args.margin, args.default_minutes)
    print(f"{len(sched.cfgs)} runs on GPUs {gpus} ({len(gpus)} in parallel): {list(sched.cfgs)}", flush=True)
    threads = [threading.Thread(target=sched.worker, args=(g,), daemon=True, name=f"gpu{g}") for g in gpus]
    for t in threads:
        t.start()
    t0 = time.time()
    while any(t.is_alive() for t in threads):
        for t in threads:
            t.join(timeout=args.every / len(threads))
        with sched.lock:
            status = dict(sched.status)
        print(f"\n{time.strftime('%H:%M:%S')} (+{(time.time() - t0) / 60:.1f} min)", flush=True)
        for name, st in status.items():
            line = last_log_line(sched.cfgs[name][1]) if st.startswith("running") else None
            print(f"  {name:<20} {st}" + (f"\n      {line}" if line else ""), flush=True)

    print("\nsummary")
    failed = []
    for name, (path, cfg) in sched.cfgs.items():
        sp = os.path.join(cfg["out_dir"], name, "summary.json")
        if os.path.exists(sp):
            s = json.load(open(sp))
            v = s.get("val_scores") or {}
            print(f"  {name:<22} val_loss {s['best_val_loss']}  R1 {v.get('rouge1_f1')}  RL {v.get('rougeL_f1')}  "
                  f"BERTScore {v.get('bertscore_f1')}  words {v.get('pred_words')}  trainable {s['params']['trainable_pct']}%  "
                  f"train {s['train_seconds'] / 60:.1f} min  peak {s['peak_mem_GB']} GB  ({sched.status[name]})")
        else:
            print(f"  {name:<20} no summary ({sched.status[name]})")
        if not sched.status[name].startswith("done"):
            failed.append(name)
    if failed:
        sys.exit(f"failed runs: {failed} (see <out_dir>/<name>.stdout)")


if __name__ == "__main__":
    main()

## Stage 2 — YOUR files (course rule 9): techniques, KD loss, experiment values

In [ ]:
%%writefile /kaggle/working/phase6/techniques/lora.py
"""YOUR FILE: LoRA (Hu et al., 2021) [3].

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); replace_linears(backbone, <which projections>, <your LoRALinear around nn.Linear>)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn
import math
from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)

class LoRALinear(nn.Module):
    def __init__(self,base:nn.Linear,r,alpha):
        super().__init__()
        self.base_layer=base
        self.in_features = base.in_features
        self.out_features = base.out_features
        for param in self.base_layer.parameters():
            param.requires_grad = False
        # define A and B as dimension B is out features cross r and A is r cross in features
        self.A = nn.Parameter(torch.randn(r, self.in_features))
        
        self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features,r))
        self.scaling  = alpha/r
    def forward (self,x):
        base_output = self.base_layer(x)
        scaled_output = (x@self.A.T)@self.B.T
        return base_output+self.scaling*scaled_output
        



def apply(backbone, r, alpha, targets):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: LoRALinear(lin, r, alpha))

In [ ]:
%%writefile /kaggle/working/phase6/techniques/lorma.py
"""YOUR FILE: LoRMA: Low-Rank Multiplicative Adaptation (Bihany, Patel, Modi, 2025) [12] -- COMPULSORY.

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); replace_linears(backbone, <which projections>, <your LoRMALinear around nn.Linear>)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn
import math
from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)

class LoRMALinear(nn.Module):
    def __init__(self,base:nn.Linear,r,alpha):
        super().__init__()
        self.base_layer=base
        self.in_features = base.in_features
        self.out_features = base.out_features
        for param in self.base_layer.parameters():
            param.requires_grad = False
        # define A and B as dimension B is out features cross r and A is r cross in features
        self.A = nn.Parameter(torch.empty(r, self.out_features))
        
        self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features,r))
        self.scaling  = alpha/r
    def forward (self,x):
        base_output = self.base_layer(x)
        scaled_output = (base_output@self.A.T)@self.B.T
        return base_output+self.scaling*scaled_output
        



def apply(backbone, r, alpha, targets):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: LoRMALinear(lin, r, alpha))

In [ ]:
%%writefile /kaggle/working/phase6/techniques/qlora.py
"""YOUR FILE: QLoRA — Dettmers et al. (2023) [9].

WHAT IT IS
  LoRA on top of a QUANTISED frozen base: the frozen W0 of every adapted linear is stored in 4 bits and dequantised
  on the fly in the forward pass; only the LoRA A/B matrices (kept in 16/32-bit) train. The paper's pieces:
    * NF4 ("4-bit NormalFloat"): 16 fixed quantisation levels placed at quantiles of a standard normal, so normally
      distributed weights use all 16 levels evenly;
    * block-wise quantisation: weights split into blocks (e.g. 64 values); each block stores its absmax as the scale,
      values stored as the index (0..15) of the nearest NF4 level after dividing by absmax;
    * double quantisation (optional): the per-block absmax constants are themselves quantised (8-bit) to save memory;
    * paged optimizers (memory paging) — not needed here.
  Forward: W0_hat = dequant(codes, absmax);  h = x @ W0_hat^T + bias + scale * B(A(x)).

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None, e.g. config {"name": "qlora", "r": ..., "alpha": ..., "targets": [...], "block_size": ...}

SUGGESTED STRUCTURE (from scratch, PyTorch only)
  NF4_LEVELS = torch.tensor([...16 values...])           # the paper / bitsandbytes NF4 table
  def quantize_nf4(w, block_size) -> (codes uint8, absmax float)   # per block: absmax, then nearest-level index
  def dequantize_nf4(codes, absmax, shape, block_size) -> tensor
  class QLoRALinear(nn.Module):
      __init__(self, base: nn.Linear, r, alpha, block_size):
          quantise base.weight once; register codes / absmax as BUFFERS (not parameters); drop the fp weight
          create LoRA A, B as in your lora.py (B = 0)
      forward(self, x): F.linear(x, dequantize(...).to(x.dtype), base.bias) + scaling * B(A(x))
  def apply(backbone, r, alpha, targets, block_size): freeze(backbone); replace_linears(... -> QLoRALinear ...)
  Note: the quantisation error of the frozen weights is part of the method; compare memory (peak_mem_GB in
  summary.json) with plain LoRA. (Using the bitsandbytes library instead would be a library implementation — if you
  do, cite it and declare it.)
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from inject import freeze, unfreeze, replace_linears   # noqa: F401

NF4_LEVELS = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367,
    -0.39491748809814453, -0.28444138169288635,
    -0.18477343022823334, -0.09105003625154495, 0.0,
    0.07958029955625534, 0.16093020141124725,
    0.24611230194568634, 0.33791524171829224,
    0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0
], dtype=torch.float32)
def quantize_nf4(w,block_size):
  assert w.numel() % block_size == 0
  blocks = w.reshape(-1,block_size)
  absmax = blocks.abs().max(dim=1).values.clamp(min=1e-12)
  blocks = blocks/absmax[:, None]
  dist  = (blocks[..., None] - NF4_LEVELS.to(blocks.device)).abs()
  codes = dist.argmin(dim=-1).to(torch.uint8)
  return (codes,absmax)
def dequantize_nf4(codes,absmax,shape,block_size):
  output = NF4_LEVELS.to(codes.device)[codes.long()]
  output = output*absmax[:, None]
  return output.reshape(shape)
class QLoRALinear(nn.Module):
  def __init__(self,base:nn.Linear,r,alpha,block_size):
    super().__init__()
    self.r = r 
    self.alpha=alpha
    self.block_size=block_size
    codes, absmax = quantize_nf4(base.weight.detach(), block_size)
    self.register_buffer("codes", codes)
    self.register_buffer("absmax", absmax)
    self.in_features=base.in_features
    self.out_features=base.out_features
    self.shape = tuple(base.weight.shape)
    if base.bias is not None:
      self.register_buffer("bias", base.bias.detach().clone())
    else:
      self.bias = None
    self.A = nn.Parameter(torch.randn(r, self.in_features))
        
    self.A = nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    self.B = nn.Parameter(torch.zeros(self.out_features,r))
    self.scaling  = alpha/r
  def forward(self,x:torch.Tensor):
    base = F.linear(x, dequantize_nf4(self.codes,self.absmax,self.shape,self.block_size).to(x.dtype), self.bias)
    lora  = (x@self.A.T)@self.B.T
    return base+self.scaling*lora











def apply(backbone, r, alpha, targets,block_size):
    freeze(backbone)
    replaced = replace_linears(backbone,
                               lambda name, lin: any(target in name for target in targets),
                               lambda name, lin: QLoRALinear(lin, r, alpha,block_size))

In [ ]:
%%writefile /kaggle/working/phase6/techniques/adalora.py

"""
AdaLoRA — Zhang et al. (2023)

SVD-shaped low-rank adaptation:
    W_delta = alpha * P @ diag(lam * mask) @ Q

Shapes:
    P: (out_features, r)
    lam: (r,)
    Q: (r, in_features)

Features:
    - Frozen pretrained base weights
    - Adaptive global rank pruning
    - EMA-based importance and uncertainty
    - Orthogonality regularization
    - Persistent pruning masks stored as buffers
"""

import torch
import torch.nn as nn

from inject import freeze, replace_linears


class AdaLoRALinear(nn.Module):
    def __init__(
        self,
        base,
        r,
        scale,
        target_rank,
        t_init,
        t_final,
        delta_t,
        beta1,
        beta2,
        orth_reg,
    ):
        super().__init__()

        self.base_layer = base
        self.in_features = base.in_features
        self.out_features = base.out_features

        if r <= 0:
            raise ValueError("r must be positive")
        if not 0 <= target_rank <= r:
            raise ValueError("target_rank must be between 0 and r")
        if delta_t <= 0:
            raise ValueError("delta_t must be positive")

        self.r = r
        self.alpha = scale / r

        self.P = nn.Parameter(
            torch.empty(self.out_features, self.r)
        )
        self.Q = nn.Parameter(
            torch.empty(self.r, self.in_features)
        )
        nn.init.xavier_normal_(self.P, gain=0.1)
        nn.init.xavier_normal_(self.Q, gain=0.1)

        # Zero initialization means the adapter initially contributes zero.
        self.lam = nn.Parameter(torch.zeros(self.r))

        self.register_buffer(
            "mask", torch.ones(self.r)
        )
        self.register_buffer(
            "ipt_ema", torch.zeros(self.r)
        )
        self.register_buffer(
            "unc_ema", torch.zeros(self.r)
        )

        self.target_rank = target_rank
        self.t_init = t_init
        self.t_final = t_final
        self.delta_t = delta_t
        self.beta1 = beta1
        self.beta2 = beta2
        self.orth_reg = orth_reg

    def forward(self, x):
        base_output = self.base_layer(x)

        # (..., in_features) -> (..., r)
        z = x @ self.Q.T

        # Masked singular values stay inactive.
        z = z * (self.lam * self.mask)

        # (..., r) -> (..., out_features)
        update = z @ self.P.T

        return base_output + self.alpha * update


def _get_modules(model):
    return [
        m for m in model.modules()
        if isinstance(m, AdaLoRALinear)
    ]


def extra_loss(model):
    """Orthogonality regularizer added to the task loss."""
    mods = _get_modules(model)

    if not mods:
        return next(model.parameters()).new_zeros(())

    loss = mods[0].P.new_zeros(())

    for m in mods:
        # Only regularize active components. This avoids penalizing
        # the geometry of components that have already been pruned.
        active = m.mask.bool()
        idx = torch.nonzero(active, as_tuple=True)[0]

        if idx.numel() == 0:
            continue

        P_active = m.P[:, idx]
        Q_active = m.Q[idx, :]

        I = torch.eye(
            idx.numel(),
            device=m.P.device,
            dtype=m.P.dtype,
        )

        loss = loss + (
            torch.norm(P_active.T @ P_active - I, p="fro") ** 2
        )
        loss = loss + (
            torch.norm(Q_active @ Q_active.T - I, p="fro") ** 2
        )

    return mods[0].orth_reg * loss


def on_step(model, step, total_steps):
    """
    Update importance EMAs and apply global rank pruning.

    Called after optimizer.step(), while gradients are still available.
    Assumes `step` uses the same convention as the training loop and
    `total_steps` is the total number of optimizer steps.
    """
    mods = _get_modules(model)

    if not mods:
        return

    # All modules should share one rank schedule.
    first = mods[0]
    r = first.r
    target_rank = first.target_rank
    t_init = first.t_init
    t_final = first.t_final
    delta_t = first.delta_t

    if any(
        m.r != r
        or m.target_rank != target_rank
        or m.t_init != t_init
        or m.t_final != t_final
        or m.delta_t != delta_t
        for m in mods
    ):
        raise ValueError(
            "All AdaLoRA modules must share rank schedule settings"
        )

    if total_steps <= t_init + t_final:
        raise ValueError(
            "total_steps must exceed t_init + t_final"
        )

    # Update importance only when all relevant gradients are available
    # and finite. If not, skip this optimizer step's AdaLoRA update.
    params = [
        p for m in mods for p in (m.P, m.Q, m.lam)
    ]

    if any(
        p.grad is None or not torch.isfinite(p.grad).all()
        for p in params
    ):
        return

    with torch.no_grad():
        # 1. Update importance and uncertainty EMAs.
        for m in mods:
            p_score = (m.P * m.P.grad).abs().mean(dim=0)
            q_score = (m.Q * m.Q.grad).abs().mean(dim=1)
            lam_score = (m.lam * m.lam.grad).abs()

            sensitivity = p_score + q_score + lam_score

            old_ipt = m.ipt_ema.clone()

            m.ipt_ema.mul_(m.beta1).add_(
                sensitivity, alpha=1.0 - m.beta1
            )

            uncertainty = (sensitivity - old_ipt).abs()

            m.unc_ema.mul_(m.beta2).add_(
                uncertainty, alpha=1.0 - m.beta2
            )

        # 2. Warm-up: keep the initial rank.
        if step < t_init:
            return

        # 3. Only prune at delta_t intervals.
        if step % delta_t != 0:
            return

        n = len(mods)
        initial_budget = r * n
        target_budget = target_rank * n

        # 4. Cubic rank-budget schedule.
        if step >= total_steps - t_final:
            budget = target_budget
        else:
            progress = (
                (step - t_init)
                / (total_steps - t_init - t_final)
            )
            progress = max(0.0, min(1.0, progress))

            scheduled_budget = (
                target_budget
                + (initial_budget - target_budget)
                * (1.0 - progress) ** 3
            )
            budget = int(scheduled_budget)

        budget = max(0, min(budget, initial_budget))

        # 5. Gather global importance scores.
        score_list = []

        for m in mods:
            score = m.ipt_ema * m.unc_ema
            score = score.masked_fill(
                m.mask == 0, float("-inf")
            )
            score_list.append(score)

        all_scores = torch.cat(score_list)

        active = int(
            sum(m.mask.sum().item() for m in mods)
        )

        # Never reactivate pruned components.
        budget = min(budget, active)

        if active <= budget:
            return

        # 6. Keep the globally highest-importance components.
        keep_idx = torch.topk(
            all_scores, k=budget
        ).indices

        keep = torch.zeros_like(
            all_scores, dtype=torch.bool
        )
        keep[keep_idx] = True

        # 7. Apply each module's portion of the global mask.
        offset = 0

        for m in mods:
            keep_slice = keep[offset:offset + m.r]

            m.mask.mul_(
                keep_slice.to(dtype=m.mask.dtype)
            )

            # Ensure pruned singular values are exactly zero.
            m.lam.mul_(m.mask)

            offset += m.r

        # Temporary diagnostic.
        if step % 200 == 0:
            active_after = int(
                sum(m.mask.sum().item() for m in mods)
            )
            print(
                f"[AdaLoRA] step={step}, "
                f"budget={budget}, active={active_after}"
            )


def apply(
    backbone,
    r,
    alpha,
    targets,
    target_rank,
    t_init,
    t_final,
    delta_t,
    beta1,
    beta2,
    orth_reg,
):
    """Inject AdaLoRA into matching linear layers."""
    freeze(backbone)

    def match(name, lin):
        return any(target in name for target in targets)

    def factory(name, lin):
        return AdaLoRALinear(
            base=lin,
            r=r,
            scale=alpha,
            target_rank=target_rank,
            t_init=t_init,
            t_final=t_final,
            delta_t=delta_t,
            beta1=beta1,
            beta2=beta2,
            orth_reg=orth_reg,
        )

    replace_linears(backbone, match, factory)

    return backbone

In [ ]:
%%writefile /kaggle/working/phase6/techniques/adapters.py
"""YOUR FILE: Houlsby / Pfeiffer bottleneck adapters [6, 7].

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); wrap_modules(backbone, <name endswith '.attn' / '.ffn'>, <your adapter wrapper>)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)

class Adapter(nn.Module):
    def __init__(self,d_model,bottleneck):
        super().__init__()
        self.d_model = d_model
        self.bottleneck=bottleneck
        self.down = nn.Linear(d_model,bottleneck)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck,d_model)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
    def forward(self,x:torch.Tensor):
        return x+self.up(self.act(self.down(x)))
class AdapterWrapper(nn.Module):
    def __init__(self,sub,d_model,bottleneck):
        super().__init__()
        self.sub=sub
        self.d_model=d_model
        self.bottleneck=bottleneck
        self.adapter =Adapter(d_model,bottleneck)
    def forward(self,*args,**kwargs):
        y= self.sub(*args,**kwargs)
        return self.adapter(y)

def apply(backbone, bottleneck):
    freeze(backbone)
    replaced = wrap_modules(
        backbone,
        lambda name, m: name.endswith((".ffn",".attn")),            # your placement choice (see below)
        lambda name, m: AdapterWrapper(m, backbone.d_model, bottleneck),
    )

In [ ]:
%%writefile /kaggle/working/phase6/techniques/prefix_tuning.py
"""YOUR FILE: Prefix Tuning — Li & Liang (2021) [4].

WHAT IT IS
  The backbone is frozen. Every self-attention layer gets P learnable key vectors and P learnable value vectors
  (a "prefix") that all queries can attend to, in addition to the real tokens. Unlike prompt tuning (input layer
  only), this steers every layer. The paper reparameterises the prefix through a small MLP during training for
  stability (prefix = MLP(embedding)), which you may or may not do.

PLUMBING HOOK ALREADY IN PLACE (phase1_models/fastattn_ab.py and fastattn_cd.py)
  If an attention module has an attribute `prefix_kv`, the fused attention calls
        pk, pv = attn.prefix_kv(batch_size)        # each (B, n_kv_heads, P, d_head)
  prepends them to that layer's keys/values (BEFORE the GQA head repetition; no RoPE is applied to them) and extends
  the attention mask so every query sees all P prefix positions (causal / bidirectional / prefix-LM rules unchanged
  for the real tokens). Without `prefix_kv` attention is exactly as before. Supported: self-attention of A/B
  (fastattn_ab), C encoder/decoder self-attention and D prefix-LM attention (fastattn_cd); not cross-attention.
  Shapes: A has n_kv_heads = n_heads = 12; B/C/D have n_kv_heads = 4; d_head = 64 everywhere.

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None, e.g. config {"name": "prefix_tuning", "n_prefix": ..., "reparam_hidden": ...}

SUGGESTED STRUCTURE
  class LayerPrefix(nn.Module):
      __init__(self, n_kv_heads, d_head, P, ...): self.k = nn.Parameter((n_kv_heads, P, d_head)), self.v = same
          (or an embedding + MLP producing them, for the reparameterised version)
      forward / __call__(self, B): return (self.k.expand(B, ...), self.v.expand(B, ...))
  def apply(backbone, n_prefix, ...):
      freeze(backbone)
      for block in backbone.blocks:                       # input -> output order; C "decoder" mode: enc + dec blocks
          attn = block.attn
          prefix = LayerPrefix(attn.n_kv_heads if hasattr(attn, "n_kv_heads") else attn.n_heads, attn.d_head, n_prefix)
          attn.prefix_module = prefix                     # register as a submodule so it is trained and saved
          attn.prefix_kv = prefix                         # the hook the fused attention calls
  IMPORTANT: assign the module as a submodule (setattr on the attention module) so its parameters are part of
  model.parameters() / state_dict; `prefix_kv` must be callable with the batch size.
"""



import torch
import torch.nn as nn

from inject import freeze, unfreeze   # noqa: F401


class LayerPrefix(nn.Module):
  def __init__(self,n_kv_heads,d_head,P):
    super().__init__()
    self.n_kv_heads = n_kv_heads
    self.d_head = d_head
    self.P=P
    self.k = nn.Parameter(torch.empty(n_kv_heads,P,d_head))
    self.v = nn.Parameter(torch.empty(n_kv_heads,P,d_head))
    nn.init.normal_(self.k,mean=0,std=0.02)
    nn.init.normal_(self.v,mean=0,std=0.02)
  def forward(self,B):
    return (self.k.unsqueeze(0).expand(B,-1,-1,-1),self.v.unsqueeze(0).expand(B,-1,-1,-1))







def apply(backbone,n_prefix):
  freeze(backbone)
  for block in backbone.blocks:
    attn=block.attn
    prefix = LayerPrefix(attn.n_kv_heads if hasattr(attn, "n_kv_heads") else attn.n_heads, attn.d_head, n_prefix)
    attn.prefix_module = prefix                     # register as a submodule so it is trained and saved
    attn.prefix_kv = prefix                         # the hook the fused attention calls

    

In [ ]:
%%writefile /kaggle/working/phase6/techniques/prompt_tuning.py
"""YOUR FILE: Prompt Tuning — Lester, Al-Rfou & Constant (2021) [5].

WHAT IT IS
  The whole backbone is frozen. P learnable "soft prompt" vectors (each of size d_model) are prepended to the
  input embeddings of every window; only those P x d_model numbers (plus the head) train.

PLUMBING HOOK ALREADY IN PLACE (backbones.Backbone)
  * backbone.n_virtual_tokens: set it to P in apply(). From then on the backbone itself prepends P placeholder ids
    (<pad>, id 0) to every window and removes the first P output positions again, so pooling / heads are unchanged.
    The plumbing checks window_len + P <= max_seq_len.
  * backbone.get_input_embedding() / backbone.set_input_embedding(module): the token embedding the prompt goes into
    (A/B/D: lm.tok_emb; C: the encoder's lm.tok_emb1).
  Your job: make positions [:, :P] of the embedding output learnable.

WHAT THE PLUMBING CALLS
  apply(backbone, **cfg) -> None, e.g. config {"name": "prompt_tuning", "n_tokens": ..., "init": ...}
      -> def apply(backbone, n_tokens, init=...):

SUGGESTED STRUCTURE
  class SoftPromptEmbedding(nn.Module):
      __init__(self, base_emb: nn.Embedding, n_tokens, init):
          keep base_emb (frozen); self.prompt = nn.Parameter(tensor of shape (n_tokens, d_model))
          init options from the paper: random (small normal) or copied from the embeddings of real vocabulary tokens
      forward(self, ids):  e = base_emb(ids)                     # (B, P + T, d), first P ids are placeholders
                           return torch.cat([self.prompt.expand(B, -1, -1).to(e.dtype), e[:, P:]], dim=1)
  def apply(backbone, n_tokens, init):
      freeze(backbone)
      backbone.set_input_embedding(SoftPromptEmbedding(backbone.get_input_embedding(), n_tokens, init))
      backbone.n_virtual_tokens = n_tokens
  Note: A/B/D tie the LM head to tok_emb; the head is not used in Phase 3 (replaced by Identity), so wrapping is safe.
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze   # noqa: F401

class SoftPromptEmbedding(nn.Module):
  def __init__(self,base_emb:nn.Embedding,n_tokens,init):
      super().__init__()
      self.base_emb = base_emb
      self.d_model = base_emb.embedding_dim
      self.n_tokens = n_tokens
     
      self.prompt = nn.Parameter(torch.empty(self.n_tokens, self.d_model))

      if init == "random":
            nn.init.normal_(self.prompt, mean=0.0, std=0.02)

      else:
            raise ValueError(f"Unknown prompt initialization: {init}")



  def forward(self,ids):
    e = self.base_emb(ids)
    B,_,_ = e.shape
    return torch.cat([self.prompt.expand(B, -1, -1).to(e.dtype), e[:, self.n_tokens:]], dim=1)
def apply(backbone,n_tokens,init):
  freeze(backbone)
  backbone.set_input_embedding(SoftPromptEmbedding(backbone.get_input_embedding(), n_tokens, init))
  backbone.n_virtual_tokens = n_tokens


    

In [ ]:
%%writefile /kaggle/working/phase6/techniques/partial_ft.py
"""YOUR FILE: Partial / layer-wise fine-tuning.

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
e.g. freeze(backbone); then unfreeze the parts you choose (backbone.blocks is input -> output order)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)


def apply(backbone,k):
    freeze(backbone)
    if(k>0):
        
    # unfreeze the final norm layer
        if(backbone.kind == "encdec"):
            if backbone.encdec_mode == "encoder":
                unfreeze(backbone.lm.ln_f_enc)
            else:
                unfreeze(backbone.lm.ln_f_dec)


            
           
        else:
            unfreeze(backbone.lm.ln_f)
        for block in  backbone.blocks[-k:]:
            unfreeze(block)
        
        
    
    




   

In [ ]:
%%writefile /kaggle/working/phase6/techniques/bitfit.py
"""YOUR FILE: BitFit (Ben Zaken et al., 2022) [8].

Contract: see techniques/__init__.py.  apply(backbone, **cfg) -> None
decide which parameters stay trainable (note: attention projections are bias=False; FFN linears and LayerNorms have biases)
cfg = the config's "technique" dict minus "name" (your options, e.g. rank / alpha / targets / ...).
"""
import torch
import torch.nn as nn

from inject import freeze, unfreeze, replace_linears, wrap_modules   # noqa: F401  (helpers, optional)


def apply(backbone, **cfg):
    freeze(backbone)
    for name , param in backbone.named_parameters():
        if name.endswith("bias") is True:
            param.requires_grad = True
        

In [ ]:
%%writefile /kaggle/working/phase6/techniques/kd.py
"""YOUR FILE: knowledge-distillation loss (Hinton et al., 2015) [2] -- COMPULSORY experiment.

Used when the config has a "kd" block: {"teacher_run": <run dir of a finished teacher>, ...your options}.
The plumbing loads the teacher (frozen, eval mode), runs it on exactly the same windows as the student and calls

kd_loss(student_logits, teacher_logits, tags, **cfg) -> scalar
    student_logits, teacher_logits: (B, T, n_tags) float32 (same tag set: checked)
    tags: (B, T) long gold tags, -100 = ignore
    cfg: the "kd" dict minus "teacher_run" (e.g. your temperature / mixing weight)
This loss REPLACES the normal task loss for the student, so include the gold-label term yourself if you want it.
The student's own adaptation is chosen as usual with "technique" (e.g. full_ft on a smaller backbone).
"""

"""YOUR FILE: knowledge-distillation loss (Hinton et al., 2015) [2]."""

import torch
import torch.nn.functional as F


def kd_loss(student_logits, teacher_logits, tags, **cfg):
    """
    Knowledge-distillation loss for token classification / NER.

    Args:
        student_logits: (B, T, C) student logits.
        teacher_logits: (B, T, C) teacher logits.
        tags: (B, T) gold tag IDs; -100 means ignore.
        cfg:
            temperature: softmax temperature (default 2.0).
            alpha: weight for KD loss (default 0.5).
                   Total loss = alpha * KD + (1-alpha) * CE.

    Returns:
        Scalar loss.
    """
    temperature = cfg.get("temperature", 2.0)
    alpha = cfg.get("alpha", 0.5)

    if temperature <= 0:
        raise ValueError("temperature must be positive")

    if not 0.0 <= alpha <= 1.0:
        raise ValueError("alpha must be between 0 and 1")

    if student_logits.shape != teacher_logits.shape:
        raise ValueError(
            "Student and teacher logits must have the same shape"
        )

    mask = tags != -100

  
    if not mask.any():
        return student_logits.sum() * 0.0

    
    teacher_logits = teacher_logits.detach()

  
    student_log_probs = F.log_softmax(
        student_logits / temperature, dim=-1
    )
    teacher_probs = F.softmax(
        teacher_logits / temperature, dim=-1
    )

    
    kl_per_token = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="none"
    ).sum(dim=-1)

    kd_term = kl_per_token[mask].mean() * (temperature ** 2)

   
    ce_term = F.cross_entropy(
        student_logits[mask],
        tags[mask],
        weight=cfg.get("weight"),
    )

   
    return alpha * kd_term + (1.0 - alpha) * ce_term

In [ ]:
%%writefile /kaggle/working/phase6/experiments.py
"""Phase-6 experiment matrix: YOUR values (course rule 9), as written by you (2026-11-02); epochs per run updated by you the same day.
max_minutes per run = safety caps added by Claude for the 12 h session scheduler (not a training choice): ~1.4 x the
estimated run time; edit freely.
Field meanings: train_summ.py / summ_infer.py docstrings."""
import copy

# ============================================================
# SHARED BASE CONFIGURATION
# ============================================================

BASE = {
    "fast_attention": True,

    "backbone": {
        "dropout": 0.0,
        "max_seq_len_override": None,
    },

    "technique": {
        "name": "full_ft",
    },

    "format": {
        "src_len": 640,
        "tgt_len": 256,
        "eos_on_truncated": True,
        "target_trunc": "sentence",
        "sep_token": "</s>",
        "start_token": None,
    },

    "selection": {
        "strategy": "headtail",
        "head_frac": 0.5,
    },

    "source": {
        "type": "document",
    },

    "split": {
        "val_docs": 500,
        "seed": 42,
    },

    "train": {
        "epochs": 5,
        "batch_examples": 8,
        "micro_examples": 1,
        "lr": 1e-4,
        "weight_decay": 0.01,
        "warmup_frac": 0.06,
        "schedule": "cosine",
        "grad_clip": 1.0,
        "seed": 42,
        "eval_every_steps": 250,
        "patience": 3,
        "max_minutes": 600,
        "log_every": 50,
    },

    "decode": {
        "length": {
            "mode": "ratio",
            "ratio": 0.25,
            "basis": "document",
            "min_words": 20,
            "max_words": 180,
            "allow_eos": True,
        },
        "temperature": 0.3,
        "top_k": 0,
        "top_p": 1.0,
        "repetition_penalty": 1.0,
        "no_repeat_ngram": 3,
        "batch_size": 16,
    },

    "eval_generate": {
        "val_docs": 100,
        "bert_model": "roberta-large",
    },

    "kd": None,
    "after": None,
}


# ============================================================
# TECHNIQUE TARGETS
# ============================================================

ATTENTION_TARGETS = [
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
]

ATTENTION_FFN_TARGETS = [
    "q_proj",
    "k_proj",
    "v_proj",
    "out_proj",
    "ffn.net.0",
    "ffn.net.2",
]


# ============================================================
# EXPERIMENT MATRIX
# ============================================================

MATRIX = {

    # R1: B2 + LoRMA
    "R1_B2_LoRMA": (
        "B2",
        {
            "technique": {
                "name": "lorma",
                "r": 8,
                "alpha": 16,
                "targets": ATTENTION_FFN_TARGETS,
            },
            "train": {
                "lr": 2e-4,
                "epochs": 10,
                "max_minutes": 160,   # safety cap set by Claude (scheduling only; edit freely)
            },
        },
    ),

    # R2: A + LoRA
    "R2_A_LoRA": (
        "A",
        {
            "technique": {
                "name": "lora",
                "r": 8,
                "alpha": 16,
                "targets": ATTENTION_TARGETS,
            },
            "train": {
                "lr": 1e-4,
                "epochs": 10,
                "max_minutes": 135,   # safety cap set by Claude (scheduling only; edit freely)
            },
        },
    ),

    # R3: B2 + QLoRA
    "R3_B2_QLoRA": (
        "B2",
        {
            "technique": {
                "name": "qlora",
                "r": 8,
                "alpha": 16,
                "targets": ATTENTION_TARGETS,
                "block_size": 64,
            },
            "train": {
                "lr": 1e-4,
                "epochs": 10,
                "max_minutes": 270,   # safety cap set by Claude (scheduling only; edit freely)
            },
        },
    ),

    # R4: C + Houlsby Adapters
    "R4_C_Houlsby": (
        "C",
        {
            "format": {
                "src_len": 384,
                "tgt_len": 256,
                "eos_on_truncated": True,
                "target_trunc": "sentence",
                "sep_token": None,
                "start_token": "last_src",
            },
            "technique": {
                "name": "adapters",
                "bottleneck": 64,
            },
            "train": {
                "lr": 1e-4,
                "epochs": 12,
                "max_minutes": 100,   # safety cap set by Claude (scheduling only; edit freely)
            },
        },
    ),

    # R5: D + Prefix Tuning
    "R5_D_Prefix": (
        "D",
        {
            "format": {
                "src_len": 624,
                "tgt_len": 256,
                "eos_on_truncated": True,
                "target_trunc": "sentence",
                "sep_token": "</s>",
                "start_token": None,
            },
            "technique": {
                "name": "prefix_tuning",
                "n_prefix": 16,
            },
            "train": {
                "lr": 5e-4,
                "epochs": 10,
                "max_minutes": 190,   # safety cap set by Claude (scheduling only; edit freely)
            },
        },
    ),

    # R6: B2 + FULL FINE-TUNING
    "R6_B2_FullFT": (
        "B2",
        {
            "technique": {
                "name": "full_ft",
            },
            "train": {
                "lr": 2e-5,
                "epochs": 8,
                "max_minutes": 130,   # safety cap set by Claude (scheduling only; edit freely)
                "batch_examples": 8,
                "micro_examples": 1,
                "weight_decay": 0.01,
                "warmup_frac": 0.06,
                "schedule": "cosine",
            },
        },
    ),

    # R7: B2 KNOWLEDGE DISTILLATION  (teacher: R1_B2_LoRMA, student: B2 with 12 layers)
    "R7_B2_KD": (
        "B2",
        {
            "backbone": {
                "truncate_layers": 12,
            },
            "technique": {
                "name": "full_ft",
            },
            "kd_teacher": "R1_B2_LoRMA",
            "kd": {
                "temperature": 2.0,
                "alpha": 0.5,
            },
            "train": {
                "lr": 2e-5,
                "epochs": 10,
                "max_minutes": 200,   # safety cap set by Claude (scheduling only; edit freely)
                "batch_examples": 8,
                "micro_examples": 1,
                "weight_decay": 0.01,
                "warmup_frac": 0.06,
                "schedule": "cosine",
            },
        },
    ),
}


# ============================================================
# KAGGLE ACCOUNT ALLOCATION
# ============================================================

SPLIT = {
    1: [
        "R1_B2_LoRMA",
        "R3_B2_QLoRA",
        "R7_B2_KD",
    ],
    2: [
        "R2_A_LoRA",
        "R4_C_Houlsby",
        "R5_D_Prefix",
        "R6_B2_FullFT",
    ],
}


# ============================================================
# CONFIG BUILDER
# ============================================================

def deep_update(d, u):
    for k, v in u.items():
        if isinstance(v, dict) and isinstance(d.get(k), dict) and k not in (
            "technique", "kd", "after"
        ):
            deep_update(d[k], v)
        else:
            d[k] = copy.deepcopy(v)
    return d


def build(phase1, data_dir, tokenizer, out_dir, cache_dir):
    out = {}
    for name, (model_key, over) in MATRIX.items():
        c = deep_update(copy.deepcopy(BASE), over)
        assert phase1.get(model_key), (
            f"{name}: Phase-1 model {model_key} not found"
        )
        c.update(
            name=name,
            model_key=model_key,
            phase1_ckpt=phase1[model_key],
            data_dir=data_dir,
            tokenizer=tokenizer,
            out_dir=out_dir,
            cache_dir=cache_dir,
        )
        out[name] = c
    return out

## Stage 3 — data + Phase-1 inputs

In [ ]:
# Phase-6 files from the gated HF dataset (Kaggle secret "HF"), Phase-1 inputs
from kaggle_secrets import UserSecretsClient
_hf, _errs = None, {}
for _label in ("HF", "HF_TOKEN"):
    try:
        _hf = UserSecretsClient().get_secret(_label)
        if _hf:
            print(f"Hugging Face token: Kaggle secret '{_label}'"); break
    except Exception as e:
        _errs[_label] = type(e).__name__
if not _hf:
    raise SystemExit(f"Could not read a Kaggle secret 'HF' or 'HF_TOKEN' ({_errs}): Add-ons -> Secrets, tick it, rerun.")
subprocess.run([sys.executable, "fetch_data.py", "--out", os.path.join(WORK, "data")], check=True,
               env=dict(os.environ, HF_TOKEN=_hf.strip()))
del _hf
DATA_DIR = os.path.join(WORK, "data", "Phase_6_Summ")

def find_all(pattern):
    return sorted(glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True))

def find_one(pattern):
    hits = find_all(pattern)
    if len(hits) > 1:
        print(f"  note: {len(hits)} matches for {pattern}, using the first: {hits}")
    return hits[0] if hits else None

def sha8(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()[:8]

TOKS = find_all("legal_bpe.json")
assert TOKS, "legal_bpe.json not found under /kaggle/input (Add Input: your Phase-1 outputs)"
tok_hashes = {t: sha8(t) for t in TOKS}
assert len(set(tok_hashes.values())) == 1, f"different legal_bpe.json files attached: {tok_hashes}"
TOKENIZER = TOKS[0]
print("tokenizer sha256:", tok_hashes[TOKENIZER], "(Phase-1 legal BPE was 38f51f14)")
PHASE1 = {"A": find_one("run_A/best.pt"), "B2": find_one("run_B2/best.pt"), "B3": find_one("ctxext/best_newctx.pt"),
          "C": find_one("run_C/best.pt"), "D": find_one("run_D/best.pt")}
for k, v in PHASE1.items():
    if v is None:
        print(f"Phase-1 {k}: NOT FOUND"); continue
    ck = torch.load(v, map_location="cpu", weights_only=False, mmap=True)
    c = ck["config"]
    lim = c["model"]["max_seq_len"] if "model" in c else (c["encoder"]["max_seq_len"], c["decoder"]["max_seq_len"])
    print(f"Phase-1 {k}: {v}\n    tag={c.get('tag')} module={c['module']} kind={c.get('kind', 'decoder')} max_len={lim}")
    del ck
import summ_data
_tr, _va, _te, DATA_REPORT = summ_data.load_splits(DATA_DIR, 0, 0)
print(DATA_REPORT)
del _tr, _va, _te

## Stage 4 — experiment queue (set ACCOUNT)

In [ ]:
# Experiment matrix = experiments.py (YOUR values). RUN = what this session trains, in priority order.
import importlib, experiments, train_summ, techniques
importlib.reload(experiments)
EXPERIMENTS = experiments.build(PHASE1, DATA_DIR, TOKENIZER, os.path.join(WORK, "runs"), os.path.join(WORK, "cache"))
ACCOUNT = None                      # 1 / 2 (this account's share, experiments.SPLIT) or "all"
assert ACCOUNT in (1, 2, "all"), "set ACCOUNT = 1 or 2 (or 'all') in this cell"
RUN = experiments.SPLIT[1] + experiments.SPLIT[2] if ACCOUNT == "all" else experiments.SPLIT[ACCOUNT]

def find_run(name):
    # a finished run (summary.json + best_model.pt) here or in an attached earlier output of this notebook
    for d in [os.path.join(WORK, "runs", name)] + sorted(glob.glob(f"/kaggle/input/**/runs/{name}", recursive=True)):
        if all(os.path.exists(os.path.join(d, f)) for f in ("summary.json", "best_model.pt", "config.json")):
            if not json.load(open(os.path.join(d, "summary.json"))).get("smoke"):
                return d
    return None

def resolve_dependencies(name, cfg, queued_before):
    # kd_teacher / stage1_from: a run queued earlier in this session, or a finished (attached) run
    for key in ("kd_teacher", "stage1_from"):
        dep = cfg.pop(key, None)
        if not dep:
            continue
        here = os.path.join(WORK, "runs", dep)
        d = here if dep in queued_before else find_run(dep)
        assert d, f"{name}: {key} {dep} is neither queued before it nor a finished attached run"
        if key == "kd_teacher":
            cfg["kd"]["teacher_run"] = d
        else:
            cfg["source"] = {"type": "stage1", **{s: os.path.join(d, f"stage1_{s}.json") for s in ("train", "val", "test")}}
            cfg["wait_for_run"] = d
    return cfg

DONE = {n: find_run(n) for n in RUN if find_run(n)}
if DONE:
    print("already finished (reused, not trained again):", DONE)
CFG_PATHS = []
for i, name in enumerate(RUN):
    if name in DONE:
        continue
    cfg = resolve_dependencies(name, EXPERIMENTS[name], [n for n in RUN[:i] if n not in DONE])
    train_summ.check_config(cfg, None)
    path = os.path.join(WORK, "configs", f"{name}.json")
    json.dump(cfg, open(path, "w"), indent=2)
    CFG_PATHS.append(path)
    f_ = cfg["format"]
    print(f"{name:24s} | {cfg['model_key']:3s} | {cfg['technique']['name']:13s} | lr {cfg['train']['lr']} | "
          f"src {f_['src_len']} tgt {f_['tgt_len']} | source {cfg['source']['type']} | kd {bool(cfg.get('kd'))} | "
          f"length {cfg['decode']['length']}")
print(f"{len(CFG_PATHS)} runs queued")

## Stage 5 — real-model smoke test

In [ ]:
# Real-model smoke test BEFORE the long runs: every queued config unchanged (real checkpoint, technique, lengths,
# micro-batch) on 48 train documents, 6 optimizer steps, validation/test generation on 4 documents, save -> reload ->
# zip. Reports peak memory, training speed and generation seconds per document. Outputs: runs_smoke/ (deleted ckpts).
RUN_SMOKE = True
if RUN_SMOKE and CFG_PATHS:
    smoke_paths = []
    for path in CFG_PATHS:
        c = json.load(open(path))
        c["smoke"] = {"steps": 6, "docs": 48}
        c["out_dir"] = os.path.join(WORK, "runs_smoke")
        c["eval_generate"] = {**(c.get("eval_generate") or {}), "val_docs": 4, "bert_model": None}
        if c.get("after"):
            c["after"]["stage1"] = {**c["after"]["stage1"], "train_docs": 4}
        if c.get("kd") and c["kd"]["teacher_run"].startswith(os.path.join(WORK, "runs")):
            c["kd"]["teacher_run"] = os.path.join(WORK, "runs_smoke", os.path.basename(c["kd"]["teacher_run"].rstrip("/")))
        if c["source"]["type"] == "stage1" and c["source"]["train"].startswith(os.path.join(WORK, "runs")):
            d = os.path.join(WORK, "runs_smoke", os.path.basename(os.path.dirname(c["source"]["train"])))
            c["source"] = {"type": "stage1", **{s: os.path.join(d, f"stage1_{s}.json") for s in ("train", "val", "test")}}
        if c["technique"]["name"] == "adalora":
            c["technique"].update(t_init=0, t_final=0, delta_t=1)
        if c.get("wait_for_run", "").startswith(os.path.join(WORK, "runs")):
            c["wait_for_run"] = os.path.join(WORK, "runs_smoke", os.path.basename(c["wait_for_run"].rstrip("/")))
        sp = os.path.join(WORK, "configs", f"smoke_{c['name']}.json")
        json.dump(c, open(sp, "w"), indent=2)
        smoke_paths.append(sp)
    t0 = time.time()
    sp_proc = subprocess.run([sys.executable, "run_many.py", "--gpus", ",".join(str(i) for i in range(max(1, N_GPUS))),
                              "--every", "60", *smoke_paths])
    print(f"\nsmoke test: {(time.time() - t0) / 60:.1f} min")
    for path in CFG_PATHS:
        c = json.load(open(path))
        sm = os.path.join(WORK, "runs_smoke", c["name"], "summary.json")
        if not os.path.exists(sm):
            print(f"  FAIL {c['name']:24s} -> {os.path.join(WORK, 'runs_smoke', c['name'] + '.stdout')}"); continue
        s_ = json.load(open(sm))
        print(f"  OK   {c['name']:24s} peak {s_['peak_mem_GB']} GB | trainable {s_['params']['trainable']:,} "
              f"({s_['params']['trainable_pct']}%) | {s_['train_tok_s']} tok/s | gen {s_['test_gen_s_per_doc']} s/doc")
    assert sp_proc.returncode == 0, "smoke test failed: fix the runs marked FAIL before launching the full runs"
    # ---- projected schedule from the MEASURED smoke speeds (upper bound: every example = src_len + tgt_len tokens) ----
    est, deps = {}, {}
    for path in CFG_PATHS:
        c = json.load(open(path)); s_ = json.load(open(os.path.join(WORK, "runs_smoke", c["name"], "summary.json")))
        f_, t_ = c["format"], c["train"]
        n_tr = DATA_REPORT["train"] - c["split"]["val_docs"]
        tok_ep = n_tr * (f_["src_len"] + f_["tgt_len"])
        train_min = t_["epochs"] * tok_ep / max(1, s_["train_tok_s"]) / 60
        steps = t_["epochs"] * -(-n_tr // t_["batch_examples"])
        n_eval = steps // (t_.get("eval_every_steps") or 10 ** 9) + t_["epochs"]
        eval_min = n_eval * c["split"]["val_docs"] * (f_["src_len"] + f_["tgt_len"]) / (3 * max(1, s_["train_tok_s"])) / 60
        gen_min = ((c.get("eval_generate") or {}).get("val_docs", 0) + DATA_REPORT["test"]) * s_["test_gen_s_per_doc"] / 60
        est[c["name"]] = dict(train=train_min, total=train_min + eval_min + gen_min + 2, cap=t_.get("max_minutes"))
        deps[c["name"]] = [os.path.basename(d.rstrip("/")) for d in
                           ([c["kd"]["teacher_run"]] if c.get("kd") else []) + ([c["wait_for_run"]] if c.get("wait_for_run") else [])]
    t_setup = (time.time() - SESSION_START) / 60 + 5
    limit = 12 * 60 - 20
    free = [t_setup] * max(1, N_GPUS); end, queue = {}, [json.load(open(p))["name"] for p in CFG_PATHS]
    while queue:
        g = min(range(len(free)), key=lambda i: free[i])
        ready = [n for n in queue if all(d in end or d not in est for d in deps[n])]
        n = ready[0] if ready else queue[0]
        start = max([free[g]] + [end[d] for d in deps[n] if d in end])
        end[n] = start + est[n]["total"]; free[g] = end[n]; queue.remove(n)
        cap = est[n]["cap"]
        warn = ""
        if cap and cap < est[n]["train"]:
            warn += f"  !! max_minutes {cap} < estimated training {est[n]['train']:.0f} min: training would be cut short"
        if cap and start + cap + 10 > limit:
            warn += f"  !! start {start:.0f} + max_minutes {cap} + 10 > {limit}: the scheduler would SKIP this run"
        print(f"  {n:24s} GPU{g} start {start / 60:4.1f} h  end {end[n] / 60:4.1f} h  (train ~{est[n]['train']:.0f} min, "
              f"total ~{est[n]['total']:.0f} min, cap {cap}){warn}")
    print(f"PROJECTED: all runs done ~{max(end.values()) / 60:.1f} h into the session (+ Stage 7 sweep ~4 min/run); "
          f"limit {limit / 60:.1f} h. Smoke speeds are measured on 6 steps (slightly pessimistic).")
    for ck in glob.glob(os.path.join(WORK, "runs_smoke", "*", "best_model.pt")):
        os.remove(ck)
    print("SMOKE TEST PASSED -> run the next cell")

## Stage 6 — full runs (2 GPUs)

In [ ]:
# Full runs: both GPUs in parallel (shared queue; a KD run waits for its teacher, a stage-2 run for its stage-1 run).
# Runs that cannot finish before the session limit (train.max_minutes + margin) are not started.
SESSION_HOURS = 12
STOP_AT = SESSION_START + SESSION_HOURS * 3600 - 20 * 60
proc = subprocess.run([sys.executable, "run_many.py", "--gpus", ",".join(str(i) for i in range(max(1, N_GPUS))),
                       "--every", "600", "--stop-at", str(STOP_AT), *CFG_PATHS])
print("exit code", proc.returncode)

## Stage 7 — decoding sweep (your settings)

In [ ]:
# Stage 7 — decoding sweep on finished runs (YOUR decode settings, course rule 9). Each entry generates validation
# summaries with summ_infer.py (both GPUs: half of the documents each), scores them (ROUGE-1/L + BERTScore) and, if
# "test": True, also writes a test submission zip. Results: sweeps/sweep_scores.jsonl.
# YOUR length caps per Phase-1 model (2026-11-02). Every finished run of this account (or attached) is re-decoded with
# its OWN decode settings, only decode.length.max_words replaced; validation = the run's full hold-out split.
MAX_WORDS = {"A": 210, "B2": 210, "C": 180, "D": 225}
VAL_LIMIT = None                    # None = all validation documents of the run's split (500); an int = first N
SWEEP = []                          # {"run", "tag", "decode", "test", "val_limit"}; extra manual entries can be appended
for _name, (_mk, _) in experiments.MATRIX.items():
    _rd = find_run(_name)
    if not _rd:
        print(f"sweep: {_name} not finished / not attached here -> skipped"); continue
    _dec = json.loads(json.dumps(json.load(open(os.path.join(_rd, "config.json")))["decode"]))
    _dec["length"]["max_words"] = MAX_WORDS[_mk]
    SWEEP.append({"run": _name, "tag": f"maxw{MAX_WORDS[_mk]}", "decode": _dec, "test": True, "val_limit": VAL_LIMIT})
print(f"{len(SWEEP)} sweep entries:", [(s_["run"], s_["tag"]) for s_ in SWEEP])
def _gen(run_dir, split, dec, out, limit=None):
    gpus = list(range(max(1, N_GPUS)))
    procs, parts = [], []
    for g in gpus:
        part = f"{out}.part{g}"
        cmd = [sys.executable, "summ_infer.py", "--run", run_dir, "--split", split, "--mode", "single",
               "--decode", json.dumps(dec), "--out", part, "--shard", f"{g}/{len(gpus)}"]
        if limit:
            cmd += ["--limit", str(limit)]
        procs.append(subprocess.Popen(cmd, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))))
        parts.append(part)
    assert all(p.wait() == 0 for p in procs), f"generation failed for {out}"
    subprocess.run([sys.executable, "summ_infer.py", "--merge", out, *parts], check=True)

import score_summ
os.makedirs("/kaggle/working/submissions", exist_ok=True)
for s in SWEEP:
    rd = find_run(s["run"])
    if not rd:
        print(f"run {s['run']} not found -> skipped"); continue
    _split = json.load(open(os.path.join(rd, "config.json")))["split"]
    base = os.path.join(WORK, "sweeps", f"{s['run']}__{s['tag']}")
    t0 = time.time()
    _gen(rd, "val", s["decode"], base + "_val.json", s.get("val_limit"))
    refs = json.load(open(os.path.join(rd, "val_refs.json"))) if not s.get("val_limit") else None
    if refs is None or len(refs) < len(json.load(open(base + "_val.json"))):
        _tr, _va, _te, _ = summ_data.load_splits(DATA_DIR, _split["val_docs"], _split.get("seed", 0))
        gen_ids = set(json.load(open(base + "_val.json")))
        refs = {d["id"]: d["summary"] for d in _va if d["id"] in gen_ids}
    preds = json.load(open(base + "_val.json"))
    res = {"run": s["run"], "tag": s["tag"], "decode": s["decode"],
           **score_summ.score(preds, refs, "roberta-large", "cuda"), "val_gen_min": round((time.time() - t0) / 60, 1)}
    if s.get("test"):
        _gen(rd, "test", s["decode"], base + "_test.json")
        _tr, _va, _te, _ = summ_data.load_splits(DATA_DIR, 0, 0)
        import summ_infer
        summ_infer.write_prediction(json.load(open(base + "_test.json")), _te, base + "_prediction.json")
        import zipfile
        zp = f"/kaggle/working/submissions/submission_{s['run']}__{s['tag']}.zip"
        with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(base + "_prediction.json", arcname="prediction.json")
        res["zip"] = zp
    print(json.dumps(res))
    with open(os.path.join(WORK, "sweeps", "sweep_scores.jsonl"), "a") as f:
        f.write(json.dumps(res) + "\n")

## Stage 8 — collect zips + table

In [ ]:
# Collect every finished run's zip + a results table (facts only)
os.makedirs("/kaggle/working/submissions", exist_ok=True)
rows = []
for sp in sorted(glob.glob(os.path.join(WORK, "runs", "*", "summary.json"))):
    s = json.load(open(sp))
    if s.get("smoke"):
        continue
    d = os.path.dirname(sp)
    for z in glob.glob(os.path.join(d, "submission_*.zip")):
        shutil.copy(z, "/kaggle/working/submissions/")
    v = s.get("val_scores") or {}
    rows.append((s["name"], s["phase1"].get("tag"), s["technique"]["name"], s["kd"], s["source"]["type"],
                 s["params"]["total"], s["params"]["trainable_pct"], s["best_val_loss"], v.get("bertscore_f1"),
                 v.get("rouge1_f1"), v.get("rougeL_f1"), v.get("pred_words"), v.get("ref_words"), v.get("rep4"),
                 round(s["train_seconds"] / 60, 1), s["peak_mem_GB"], s["test_gen_s_per_doc"]))
hdr = ("run", "phase1", "technique", "kd", "source", "params", "trainable%", "val_loss", "BERTScore", "R1", "RL",
       "words", "ref_words", "rep4", "train_min", "peak_GB", "gen_s/doc")
print("\t".join(hdr))
for r in rows:
    print("\t".join(str(x) for x in r))
print(sorted(os.listdir("/kaggle/working/submissions")))